# CS781 Phase 3 — Experiment 5
## Phase-1 Model 4 + AdaLoRA — Rhetorical Role Prediction

This experiment returns to the strongest Phase-1 backbone, **Model 4**, but uses a substantially different adaptation technique from Exp1 plain LoRA.

### Experiment identity
**Phase-1 Model 4 + custom AdaLoRA-style adaptive low-rank allocation**

The exact Phase-1 Model 4 checkpoint is loaded strictly and frozen. AdaLoRA adapters are inserted into:

- attention: `q_proj`, `v_proj`, `o_proj`
- SwiGLU MLP: `gate_proj`, `up_proj`, `down_proj`

`k_proj` is intentionally excluded because Model 4 uses GQA and K has only 128 output dimensions; the adaptive rank budget is spent on the larger / more task-sensitive projections.

### Adaptive rank mechanism
Each adapted projection starts at rank `r_init=16`. During training, the notebook:

1. tracks an EMA importance score for every rank component using `|e * grad(e)|`;
2. gradually shrinks the **global active-rank budget**;
3. permanently masks the lowest-importance rank components;
4. finishes at an average rank budget of approximately `r_target=8` per adapted projection.

This is not fixed-rank LoRA: different layers/projections may finish with different active ranks depending on learned importance.

### Phase-3 contract
- train: `CL_train + IT_train`
- dev selection: combined 13-label Macro-F1 on `CL_dev + IT_dev`
- exact supplied evaluator must pass before refit
- dev-only transition decoding
- fresh train+dev refit replays the selected fractional training position
- heldout, refit, and heldout+refit ensemble submissions
- strict private-test structural checks
- `prediction.json` at ZIP root

### Runtime / reproducibility
- two dev evaluations per epoch to catch early peaks
- deterministic sortish length grouping
- AMP + fused AdamW when available
- live elapsed / ETA
- exact adapter-rank statistics saved in checkpoints and manifest

> Attach `eff_full_best*.pt` and the exact Phase-1 `tokenizer.json`, keep `HF_TOKEN` in Kaggle Secrets, and run top-to-bottom.


### v2 changes (aimed at the best F1 for Model 4)
- **Bug fix — importance poisoning under fp16.** Importance was updated even on steps where the AMP GradScaler found inf/NaN gradients (common in the first steps, scale 65536). One bad step made a component's EMA importance inf/NaN, which breaks the global ranking used for pruning. Non-finite scores are now skipped.
- **AdaLoRA-style sensitivity.** Importance is now `|e·∇e| + mean|A·∇A| + mean|B·∇B|` per rank component (the paper's singular-value + singular-vector sensitivity), not just the scalar `e`.
- **Pruning finishes inside the productive region.** Exp1 peaked within the first epoch, so the old 6-epoch schedule would barely start pruning before the best checkpoint. Now 3 epochs, pruning from 5% to 60% of training.
- **Capacity matched to Exp1:** `r_init=16` (same as the LoRA run), pruned to an average of 8; `alpha=32` keeps the same scale (2.0).
- **Regularisation against the quick overfit seen in Exp1:** adapter dropout 0.10, label smoothing 0.03.
- **Finer model selection:** 4 dev evaluations per epoch (aligned to optimizer updates), patience 6 evaluations.
- **Speed:** micro-batch 8 × accumulation 2 (same effective batch 16).
- **KD export cell (new, last cell):** saves the refit model's logits for every train+dev sentence to `teacher_logits_train_dev.npz`, so this Model 4 model can serve as the teacher for the mandatory knowledge-distillation experiment.
- Honest scope note for the report: this is an *AdaLoRA-style* implementation (importance-based global rank pruning); it does not include the paper's orthogonality regulariser.


### v3 review/fixes
I kept the strong v2 changes: `r_init=16 -> r_target=8`, 3-epoch adaptive schedule,
4 dev evaluations/epoch, label smoothing, 8×2 effective batch 16, finite-safe
AdaLoRA sensitivity, and KD-logit export.

v3 adds robustness without changing the experiment identity:
- rank-budget computation is now **model-specific**, so heldout/refit allocation cannot accidentally depend on a global model object;
- global pruning preserves a small **minimum rank per adapted projection**, preventing one projection from being completely zeroed by noisy early importance;
- rank allocation asserts finite importance before sorting and reports min/mean/max active rank;
- checkpoint/manifest record the actual rank distribution at the selected best point;
- KD export now saves **heldout teacher logits on train only** and **refit teacher logits on train+dev** separately, which is cleaner for the mandatory KD experiment.


In [1]:
# ============================================================
# CELL 1 — Imports, seed, paths, and Experiment-5 config
# ============================================================
import os, re, gc, json, math, time, random, zipfile, subprocess, sys, hashlib
from pathlib import Path
from collections import Counter

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset
from sklearn.metrics import f1_score, precision_score, recall_score, classification_report
from huggingface_hub import HfApi, hf_hub_download, login

SEED = 1234
def seed_everything(seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
seed_everything()

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
torch.set_float32_matmul_precision("high")

HF_TOKEN = None
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
except Exception as e:
    print("Could not read Kaggle HF_TOKEN secret:", repr(e))
assert HF_TOKEN, "HF_TOKEN not found in Kaggle Secrets."
login(token=HF_TOKEN)
HF_REPO = "Exploration-Lab/CS-781-Capstone"

ROOT = Path("/kaggle/working/phase3_exp5_model4_adalora")
ROOT.mkdir(parents=True, exist_ok=True)

BEST_CKPT = ROOT / "best_adalora_rr.pt"
FINAL_CKPT = ROOT / "final_refit_adalora_rr.pt"

DEV_PRED = ROOT / "prediction_dev.json"
CL_DEV_TRUTH = ROOT / "cl_dev_truth.json"
IT_DEV_TRUTH = ROOT / "it_dev_truth.json"
LABELS_JSON = ROOT / "labels.json"
EVAL_SCRIPT = ROOT / "evaluate_phase3.py"

OFFICIAL_PRED = ROOT / "prediction.json"
SUBMISSION_ZIP = ROOT / "submission.zip"

# -------- exact Phase-1 Model 4 --------
VOCAB_SIZE = 24_576
D_MODEL = 640
N_LAYERS = 17
N_HEADS = 10
N_KV_HEADS = 2
FFN_DIM = 2048
MAX_POSITIONS = 2560
ROPE_BASE = 10_000.0
RMS_EPS = 1e-5
DROPOUT = 0.0
EXPECTED_TOKENIZER_SHA256 = "9cfbe6f0e49067f63ba71fe1013021c5de9269a2b4bc6c9ae30a72e42102106d"

# -------- Phase-3 context --------
MAX_LEN = 640
CONTEXT_SENTS = 2
USE_POS = True
TARGET_BUDGET = 320
PREV_BUDGET = 120
NEXT_BUDGET = 120

# -------- AdaLoRA --------
ADALORA_TARGETS = [
    "q_proj", "v_proj", "o_proj",
    "gate_proj", "up_proj", "down_proj"
]
ADALORA_R_INIT = 16
ADALORA_R_TARGET = 8
ADALORA_ALPHA = 32
ADALORA_DROPOUT = 0.10
ADALORA_LR = 2e-4

# Rank allocation schedule:
# no pruning until TINIT_FRAC; reach final budget at TFINAL_FRAC.
ADALORA_TINIT_FRAC = 0.05
ADALORA_TFINAL_FRAC = 0.60
ADALORA_PRUNE_INTERVAL = 50
ADALORA_IMPORTANCE_BETA = 0.90
ADALORA_MIN_RANK_PER_MODULE = 2  # safety floor; avoids fully disabling a projection

# -------- RR head --------
HEAD_DROPOUT = 0.15
HEAD_LR = 1e-3

# -------- training --------
NUM_EPOCHS = 3
EVAL_PER_EPOCH = 4
EARLY_STOP_PATIENCE_EVALS = 6
LABEL_SMOOTHING = 0.03
MICRO_BATCH_SIZE = 8
EVAL_BATCH_SIZE = 16
GRAD_ACCUM_STEPS = 2
WEIGHT_DECAY = 0.01
GRAD_CLIP = 1.0
WARMUP_RATIO = 0.06
USE_AMP = True

CLASS_WEIGHT_MIN = 0.35
CLASS_WEIGHT_MAX = 3.50
PRINT_EVERY = 250
NUM_WORKERS = 2
LENGTH_GROUPED_TRAINING = True
LENGTH_BUCKET_MULT = 64
NAN_CHECK_EVERY = 50

REFIT_MATCH_ORIGINAL_SCHEDULE = True
ENSEMBLE_HELDOUT_AND_REFIT = True

print("Output directory:", ROOT)
print("Model4:", N_LAYERS, "layers | d=", D_MODEL, "|", N_HEADS, "Q /", N_KV_HEADS, "KV")
print("AdaLoRA targets:", ADALORA_TARGETS)
print(
    "AdaLoRA rank:",
    ADALORA_R_INIT, "->", ADALORA_R_TARGET,
    "| alpha=", ADALORA_ALPHA,
    "| prune interval=", ADALORA_PRUNE_INTERVAL
)
print("Context sentences each side:", CONTEXT_SENTS, "| MAX_LEN:", MAX_LEN)


Device: cuda
GPU: Tesla T4
Output directory: /kaggle/working/phase3_exp5_model4_adalora
Model4: 17 layers | d= 640 | 10 Q / 2 KV
AdaLoRA targets: ['q_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']
AdaLoRA rank: 16 -> 8 | alpha= 32 | prune interval= 50
Context sentences each side: 2 | MAX_LEN: 640


In [2]:
# ============================================================
# CELL 2 — Find Phase-1 Model 4 checkpoint and tokenizer safely
# ============================================================
def find_files(predicate):
    out=[]
    for base in ["/kaggle/input", "/kaggle/working"]:
        if not os.path.isdir(base): continue
        for root, _, files in os.walk(base):
            for fn in files:
                p=os.path.join(root,fn)
                if predicate(fn,p): out.append(p)
    return out

ckpt_matches = find_files(
    lambda fn,p: fn.lower().endswith(".pt") and "eff_full_best" in fn.lower()
)
tok_matches = find_files(
    lambda fn,p: fn.lower().endswith(".json") and "tokenizer" in fn.lower()
)

print("Checkpoint candidates:")
for p in ckpt_matches: print(" ",p)
print("Tokenizer candidates:")
for p in tok_matches: print(" ",p)

assert ckpt_matches, "Attach the Phase-1 Model4 best checkpoint (eff_full_best*.pt)."
assert tok_matches, "Attach the exact Phase-1 tokenizer.json."

def sha256_file(p):
    h=hashlib.sha256()
    with open(p,"rb") as f:
        for b in iter(lambda:f.read(1<<20),b""):
            h.update(b)
    return h.hexdigest()

def score_path(p,words):
    low=p.lower()
    return sum(w in low for w in words)

PHASE1_CKPT=sorted(
    ckpt_matches,
    key=lambda p:(score_path(p,["model4","efficient","phase1","eff_full_best"]),p),
    reverse=True
)[0]

exact_tok=[p for p in tok_matches if sha256_file(p)==EXPECTED_TOKENIZER_SHA256]
TOKENIZER_FILE=exact_tok[0] if exact_tok else sorted(tok_matches)[0]

print("Using checkpoint:",PHASE1_CKPT)
print("Using tokenizer :",TOKENIZER_FILE)
print("Tokenizer SHA256:",sha256_file(TOKENIZER_FILE))

assert sha256_file(TOKENIZER_FILE)==EXPECTED_TOKENIZER_SHA256, (
    "Tokenizer hash mismatch — attach the exact Phase-1 tokenizer."
)


Checkpoint candidates:
  /kaggle/input/datasets/aaryant23/phase2-model4/eff_full_best (6).pt
Tokenizer candidates:
  /kaggle/input/datasets/aaryant23/phase2-model4/tokenizer.json
Using checkpoint: /kaggle/input/datasets/aaryant23/phase2-model4/eff_full_best (6).pt
Using tokenizer : /kaggle/input/datasets/aaryant23/phase2-model4/tokenizer.json
Tokenizer SHA256: 9cfbe6f0e49067f63ba71fe1013021c5de9269a2b4bc6c9ae30a72e42102106d


In [3]:
# ============================================================
# CELL 3 — Original custom byte-level LegalBPE
# ============================================================

BPE_SPECIALS = ["<pad>", "<unk>", "<s>", "</s>"]

class LegalBPE:
    def __init__(self, vocab_size=5000):
        self.vocab_size = int(vocab_size)
        self.merges = []
        self.itos = list(BPE_SPECIALS)
        self.stoi = {t:i for i,t in enumerate(self.itos)}
        self.symbol_bytes = {}
        for b in range(256):
            s = f"{b:02x}"
            self.symbol_bytes[s] = bytes([b])
            self.stoi[s] = len(self.itos)
            self.itos.append(s)
        self._reset_encode_cache()

    @property
    def pad_token_id(self): return self.stoi["<pad>"]
    @property
    def unk_token_id(self): return self.stoi["<unk>"]
    @property
    def bos_token_id(self): return self.stoi["<s>"]
    @property
    def eos_token_id(self): return self.stoi["</s>"]
    def __len__(self): return len(self.itos)

    @staticmethod
    def word_to_symbols(word):
        return [f"{b:02x}" for b in word.encode("utf-8")]

    def _reset_encode_cache(self):
        self._ranks = {(l,r): i for i,(l,r) in enumerate(self.merges)}
        self._word_cache = {}
        self._cache_nmerges = len(self.merges)

    def encode_word(self, word):
        if self._cache_nmerges != len(self.merges):
            self._reset_encode_cache()

        hit = self._word_cache.get(word)
        if hit is not None:
            return list(hit)

        symbols = self.word_to_symbols(word)
        ranks = self._ranks

        while len(symbols) > 1:
            best = None
            best_rank = None
            for j in range(len(symbols)-1):
                r = ranks.get((symbols[j], symbols[j+1]))
                if r is not None and (best_rank is None or r < best_rank):
                    best = (symbols[j], symbols[j+1])
                    best_rank = r
            if best is None:
                break

            left, right = best
            merged = []
            j = 0
            while j < len(symbols):
                if j+1 < len(symbols) and symbols[j] == left and symbols[j+1] == right:
                    merged.append(left+right)
                    j += 2
                else:
                    merged.append(symbols[j])
                    j += 1
            symbols = merged

        if len(self._word_cache) >= 250_000:
            self._word_cache.clear()
        self._word_cache[word] = tuple(symbols)
        return symbols

    def encode(self, text):
        out = []
        unk = self.unk_token_id
        for unit in re.findall(r"\S+|\s+", text):
            for s in self.encode_word(unit):
                out.append(self.stoi.get(s, unk))
        return out

    def decode(self, ids):
        buf = bytearray()
        for idx in ids:
            idx = int(idx)
            tok = self.itos[idx]
            if tok in BPE_SPECIALS:
                continue
            buf.extend(self.symbol_bytes[tok])
        return bytes(buf).decode("utf-8", errors="replace")

    @classmethod
    def load_json(cls, path):
        obj = json.load(open(path, encoding="utf-8"))
        tok = cls(obj["vocab_size"])
        tok.merges = [list(x) for x in obj["merges"]]
        tok.itos = list(obj["itos"])
        tok.stoi = {t:i for i,t in enumerate(tok.itos)}
        tok.vocab_size = len(tok.itos)

        tok.symbol_bytes = {}
        for b in range(256):
            s = f"{b:02x}"
            tok.symbol_bytes[s] = bytes([b])
        for l, r in tok.merges:
            tok.symbol_bytes[l+r] = tok.symbol_bytes[l] + tok.symbol_bytes[r]

        tok._reset_encode_cache()
        return tok

tokenizer = LegalBPE.load_json(TOKENIZER_FILE)
print("Tokenizer vocab:", len(tokenizer))
assert len(tokenizer) == VOCAB_SIZE, (len(tokenizer), VOCAB_SIZE)

probe = "IN THE SUPREME COURT OF INDIA\nSection 14: legal test."
assert tokenizer.decode(tokenizer.encode(probe)) == probe
print("✅ tokenizer round-trip passed")

Tokenizer vocab: 24576
✅ tokenizer round-trip passed


In [4]:
# ============================================================
# CELL 4 — Exact Phase-1 Model 4 architecture
# ============================================================

class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-5):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(dim))
        self.eps = eps

    def forward(self, x):
        dtype = x.dtype
        x32 = x.float()
        x32 = x32 * torch.rsqrt(x32.pow(2).mean(-1, keepdim=True) + self.eps)
        return x32.to(dtype) * self.weight


class RotaryTable(nn.Module):
    def __init__(self, head_dim, max_positions, base=10000.0):
        super().__init__()
        assert head_dim % 2 == 0
        inv_freq = 1.0 / (base ** (torch.arange(0, head_dim, 2, dtype=torch.float32) / head_dim))
        freqs = torch.outer(torch.arange(max_positions, dtype=torch.float32), inv_freq)
        self.register_buffer("cos_cached", freqs.cos(), persistent=False)
        self.register_buffer("sin_cached", freqs.sin(), persistent=False)
        self.max_positions = max_positions

    def forward(self, position_ids):
        return (
            self.cos_cached[position_ids].unsqueeze(1),
            self.sin_cached[position_ids].unsqueeze(1),
        )


def apply_rope(x, cos, sin):
    B, H, T, D = x.shape
    cos = cos.to(x.dtype)
    sin = sin.to(x.dtype)
    xp = x.view(B, H, T, D // 2, 2)
    x0, x1 = xp[..., 0], xp[..., 1]
    return torch.stack(
        (x0 * cos - x1 * sin, x0 * sin + x1 * cos),
        dim=-1
    ).flatten(-2)


class GQACausalSelfAttention(nn.Module):
    GQA_DECODE_PATH = True

    def __init__(self, d_model, n_heads, n_kv_heads, dropout):
        super().__init__()
        assert d_model % n_heads == 0
        assert n_heads % n_kv_heads == 0
        self.n_heads = n_heads
        self.n_kv_heads = n_kv_heads
        self.head_dim = d_model // n_heads
        self.kv_repeat = n_heads // n_kv_heads
        self.dropout = dropout

        self.q_proj = nn.Linear(d_model, n_heads * self.head_dim, bias=False)
        self.k_proj = nn.Linear(d_model, n_kv_heads * self.head_dim, bias=False)
        self.v_proj = nn.Linear(d_model, n_kv_heads * self.head_dim, bias=False)
        self.o_proj = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x, cos, sin, past_kv=None, use_cache=False, key_valid=None):
        B, T, C = x.shape
        H, KV, D = self.n_heads, self.n_kv_heads, self.head_dim

        q = self.q_proj(x).view(B, T, H, D).transpose(1, 2)
        k = self.k_proj(x).view(B, T, KV, D).transpose(1, 2)
        v = self.v_proj(x).view(B, T, KV, D).transpose(1, 2)

        q = apply_rope(q, cos, sin)
        k = apply_rope(k, cos, sin)

        past_len = 0
        if past_kv is not None:
            pk, pv = past_kv
            past_len = pk.size(2)
            k = torch.cat([pk, k], dim=2)
            v = torch.cat([pv, v], dim=2)

        new_kv = (k, v) if use_cache else None
        S = k.size(2)
        drop = self.dropout if self.training else 0.0

        if T == 1 and past_len > 0 and self.GQA_DECODE_PATH:
            qg = q.reshape(B, KV, self.kv_repeat, D)
            mask = None if key_valid is None else key_valid[:, None, None, :]
            y = F.scaled_dot_product_attention(
                qg, k, v, attn_mask=mask, dropout_p=drop, is_causal=False
            )
            y = y.reshape(B, H, 1, D).transpose(1, 2).contiguous().view(B, 1, C)
            return self.o_proj(y), new_kv

        if self.kv_repeat != 1:
            k_attn = k.repeat_interleave(self.kv_repeat, dim=1)
            v_attn = v.repeat_interleave(self.kv_repeat, dim=1)
        else:
            k_attn, v_attn = k, v

        attn_mask, is_causal = None, False
        if T == 1:
            if key_valid is not None:
                attn_mask = key_valid[:, None, None, :]
        elif key_valid is None and past_len == 0:
            is_causal = True
        else:
            qi = torch.arange(T, device=x.device)[:, None] + past_len
            kj = torch.arange(S, device=x.device)[None, :]
            attn_mask = (kj <= qi)[None, None, :, :]
            if key_valid is not None:
                attn_mask = attn_mask & key_valid[:, None, None, :]
                if past_len == 0:
                    attn_mask = attn_mask | (kj == qi)[None, None, :, :]

        y = F.scaled_dot_product_attention(
            q, k_attn, v_attn,
            attn_mask=attn_mask,
            dropout_p=drop,
            is_causal=is_causal
        )
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.o_proj(y), new_kv


class SwiGLU(nn.Module):
    def __init__(self, d_model, hidden_dim, dropout):
        super().__init__()
        self.gate_proj = nn.Linear(d_model, hidden_dim, bias=False)
        self.up_proj = nn.Linear(d_model, hidden_dim, bias=False)
        self.down_proj = nn.Linear(hidden_dim, d_model, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        x = F.silu(self.gate_proj(x)) * self.up_proj(x)
        return self.dropout(self.down_proj(x))


class EfficientDecoderBlock(nn.Module):
    def __init__(self, d_model, n_heads, n_kv_heads, ffn_dim, dropout, rms_eps):
        super().__init__()
        self.norm1 = RMSNorm(d_model, rms_eps)
        self.attn = GQACausalSelfAttention(d_model, n_heads, n_kv_heads, dropout)
        self.norm2 = RMSNorm(d_model, rms_eps)
        self.ff = SwiGLU(d_model, ffn_dim, dropout)
        self.resid_drop = nn.Dropout(dropout)

    def forward(self, x, cos, sin, past_kv=None, use_cache=False, key_valid=None):
        a, new_kv = self.attn(
            self.norm1(x), cos, sin,
            past_kv=past_kv,
            use_cache=use_cache,
            key_valid=key_valid
        )
        x = x + self.resid_drop(a)
        x = x + self.ff(self.norm2(x))
        return x, new_kv


class EfficientDecoderLM(nn.Module):
    def __init__(
        self, vocab_size, d_model, n_layers, n_heads, n_kv_heads,
        ffn_dim, dropout, max_positions, rope_base, rms_eps
    ):
        super().__init__()
        self.vocab_size = vocab_size
        self.d_model = d_model
        self.max_positions = max_positions

        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.drop = nn.Dropout(dropout)
        self.rope = RotaryTable(d_model // n_heads, max_positions, rope_base)

        self.blocks = nn.ModuleList([
            EfficientDecoderBlock(
                d_model, n_heads, n_kv_heads, ffn_dim, dropout, rms_eps
            )
            for _ in range(n_layers)
        ])
        self.final_norm = RMSNorm(d_model, rms_eps)

        self.apply(self._init_weights)
        scaled_std = 0.02 / math.sqrt(2 * n_layers)
        for block in self.blocks:
            nn.init.normal_(block.attn.o_proj.weight, mean=0.0, std=scaled_std)
            nn.init.normal_(block.ff.down_proj.weight, mean=0.0, std=scaled_std)

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)

    def forward_hidden(
        self, input_ids, past_key_values=None, use_cache=False,
        position_ids=None, key_valid=None
    ):
        B, T = input_ids.shape
        past_len = 0 if past_key_values is None else past_key_values[0][0].size(2)

        if past_len + T > self.max_positions:
            raise ValueError(
                f"context {past_len + T} exceeds max_positions={self.max_positions}"
            )

        if position_ids is None:
            position_ids = torch.arange(
                past_len, past_len + T, device=input_ids.device
            )[None, :].expand(B, T)

        cos, sin = self.rope(position_ids)

        x = self.drop(self.token_emb(input_ids))
        new_cache = [] if use_cache else None

        for i, block in enumerate(self.blocks):
            past = None if past_key_values is None else past_key_values[i]
            x, kv = block(
                x, cos, sin,
                past_kv=past,
                use_cache=use_cache,
                key_valid=key_valid
            )
            if use_cache:
                new_cache.append(kv)

        return self.final_norm(x), new_cache

    def project(self, hidden):
        return F.linear(hidden, self.token_emb.weight)


def build_backbone():
    return EfficientDecoderLM(
        vocab_size=len(tokenizer),
        d_model=D_MODEL,
        n_layers=N_LAYERS,
        n_heads=N_HEADS,
        n_kv_heads=N_KV_HEADS,
        ffn_dim=FFN_DIM,
        dropout=DROPOUT,
        max_positions=MAX_POSITIONS,
        rope_base=ROPE_BASE,
        rms_eps=RMS_EPS,
    )

backbone = build_backbone()
print("Fresh backbone params:", f"{sum(p.numel() for p in backbone.parameters())/1e6:.3f}M")

Fresh backbone params: 99.309M


In [5]:
# ============================================================
# CELL 5 — Load Phase-1 Model 4 checkpoint exactly
# ============================================================

phase1_obj = torch.load(PHASE1_CKPT, map_location="cpu", weights_only=False)
print("Checkpoint keys:", list(phase1_obj.keys()))

state = phase1_obj["model"] if isinstance(phase1_obj, dict) and "model" in phase1_obj else phase1_obj

missing, unexpected = backbone.load_state_dict(state, strict=True)
assert not missing and not unexpected

print("✅ Phase-1 Model 4 weights loaded with strict=True")
print("global_step:", phase1_obj.get("global_step") if isinstance(phase1_obj, dict) else None)
print("val_loss:", phase1_obj.get("val_loss") if isinstance(phase1_obj, dict) else None)
del state
gc.collect()

Checkpoint keys: ['model', 'global_step', 'val_loss', 'config']
✅ Phase-1 Model 4 weights loaded with strict=True
global_step: 16266
val_loss: 1.3664915735571412


63

In [6]:
# ============================================================
# CELL 6 — Custom AdaLoRA from scratch
# Adaptive global rank allocation over Model4 projections
# ============================================================
class AdaLoRALinear(nn.Module):
    """
    Frozen base Linear + adaptive low-rank residual.

    delta(x) = scale * B( (A(dropout(x))) * e * mask )

    A rank component j is controlled by scalar e[j].
    `adalora_mask[j]` is permanently set to 0 when that component is pruned.

    Importance is an EMA of |e * grad(e)|. The global allocator keeps the
    highest-importance active components as the rank budget shrinks.
    """
    def __init__(self,base_linear,r_init,alpha,dropout):
        super().__init__()
        assert isinstance(base_linear,nn.Linear)
        assert r_init>0

        self.base=base_linear
        for p in self.base.parameters():
            p.requires_grad=False

        self.r_init=int(r_init)
        self.alpha=float(alpha)
        self.scaling=self.alpha/self.r_init
        self.dropout=nn.Dropout(dropout)

        self.adalora_A=nn.Parameter(
            torch.empty(self.r_init,base_linear.in_features)
        )
        self.adalora_B=nn.Parameter(
            torch.zeros(base_linear.out_features,self.r_init)
        )
        self.adalora_e=nn.Parameter(torch.ones(self.r_init))

        nn.init.kaiming_uniform_(self.adalora_A,a=math.sqrt(5))
        nn.init.zeros_(self.adalora_B)
        nn.init.ones_(self.adalora_e)

        self.register_buffer(
            "adalora_mask",
            torch.ones(self.r_init,dtype=torch.float32),
            persistent=True,
        )
        self.register_buffer(
            "adalora_importance",
            torch.zeros(self.r_init,dtype=torch.float32),
            persistent=True,
        )

    def forward(self,x):
        base_out=self.base(x)
        z=F.linear(self.dropout(x),self.adalora_A)
        z=z*(self.adalora_e*self.adalora_mask).to(z.dtype)
        delta=F.linear(z,self.adalora_B)
        return base_out+self.scaling*delta

    @torch.no_grad()
    def update_importance(self,beta=ADALORA_IMPORTANCE_BETA):
        ge,gA,gB=self.adalora_e.grad,self.adalora_A.grad,self.adalora_B.grad
        if ge is None or gA is None or gB is None:
            return
        # AdaLoRA-style sensitivity of the triplet (singular value, A row, B column).
        s_e=(self.adalora_e.detach()*ge.detach()).abs().float()
        s_A=(self.adalora_A.detach()*gA.detach()).abs().float().mean(dim=1)
        s_B=(self.adalora_B.detach()*gB.detach()).abs().float().mean(dim=0)
        score=s_e+s_A+s_B
        # fp16 overflow step (GradScaler will skip it): never let inf/NaN enter the EMA.
        if not bool(torch.isfinite(score).all()):
            return
        self.adalora_importance.mul_(beta).add_(score,alpha=1.0-beta)

    def active_rank(self):
        return int(self.adalora_mask.sum().item())


def freeze_all(module):
    for p in module.parameters():
        p.requires_grad=False


def inject_adalora(
    model,
    targets=ADALORA_TARGETS,
    r_init=ADALORA_R_INIT,
    alpha=ADALORA_ALPHA,
    dropout=ADALORA_DROPOUT,
):
    freeze_all(model)
    replaced=[]
    for i,block in enumerate(model.blocks):
        groups=(
            ("attn",block.attn,["q_proj","k_proj","v_proj","o_proj"]),
            ("ff",block.ff,["gate_proj","up_proj","down_proj"]),
        )
        for gname,mod,names in groups:
            for nme in names:
                if nme in targets:
                    setattr(
                        mod,nme,
                        AdaLoRALinear(
                            getattr(mod,nme),
                            r_init=r_init,
                            alpha=alpha,
                            dropout=dropout,
                        )
                    )
                    replaced.append(f"blocks.{i}.{gname}.{nme}")
    return replaced


def iter_adalora_modules(m):
    for name,mod in m.named_modules():
        if isinstance(mod,AdaLoRALinear):
            yield name,mod


@torch.no_grad()
def adalora_rank_summary(m):
    rows=[]
    for name,mod in iter_adalora_modules(m):
        rows.append((name,mod.active_rank(),mod.r_init))
    active=sum(x[1] for x in rows)
    maximum=sum(x[2] for x in rows)
    ranks=[x[1] for x in rows]
    stats={
        "min": int(min(ranks)) if ranks else 0,
        "max": int(max(ranks)) if ranks else 0,
        "mean": float(np.mean(ranks)) if ranks else 0.0,
    }
    return rows,active,maximum,stats


@torch.no_grad()
def adalora_update_importance(m):
    for _,mod in iter_adalora_modules(m):
        mod.update_importance()


@torch.no_grad()
def adalora_prune_to_budget(m,target_active,min_rank_per_module=ADALORA_MIN_RANK_PER_MODULE):
    """
    Global rank allocator with a small per-module safety floor.
    Lowest-importance active components are pruned first, but each adapted
    projection keeps at least `min_rank_per_module` active components.
    """
    modules=list(iter_adalora_modules(m))
    current=sum(mod.active_rank() for _,mod in modules)
    hard_floor=sum(min(min_rank_per_module,mod.r_init) for _,mod in modules)
    target_active=int(max(hard_floor,min(int(target_active),current)))

    n_prune=current-target_active
    if n_prune<=0:
        return 0

    candidates=[]
    for mi,(name,mod) in enumerate(modules):
        active=torch.nonzero(mod.adalora_mask>0.5,as_tuple=False).flatten()
        removable=max(0,len(active)-min_rank_per_module)
        if removable<=0:
            continue

        vals=[]
        for j in active.tolist():
            imp=float(mod.adalora_importance[j].item())
            if not math.isfinite(imp):
                imp=float("inf")  # never prune on corrupted/unknown importance
            vals.append((imp,int(j)))

        # Only components above the module floor are eligible globally.
        vals.sort(key=lambda x:(x[0],x[1]))
        for imp,j in vals[:removable]:
            candidates.append((imp,mi,j,name))

    candidates.sort(key=lambda x:(x[0],x[3],x[2]))
    actually=min(n_prune,len(candidates))
    for _,mi,j,_ in candidates[:actually]:
        modules[mi][1].adalora_mask[j]=0.0

    return actually


def rank_budget_for_update(m,step,total_updates):
    n_modules=sum(1 for _ in iter_adalora_modules(m))
    start=n_modules*ADALORA_R_INIT
    final=n_modules*ADALORA_R_TARGET

    t_init=max(1,int(total_updates*ADALORA_TINIT_FRAC))
    t_final=max(t_init+1,int(total_updates*ADALORA_TFINAL_FRAC))

    if step<=t_init:
        return start
    if step>=t_final:
        return final

    # Cubic decay is deliberately gradual early and stronger near the end.
    p=(step-t_init)/(t_final-t_init)
    frac=(1.0-p)**3
    return int(round(final+(start-final)*frac))


replaced=inject_adalora(backbone)
print("Injected AdaLoRA modules:",len(replaced))
print("First 8:",replaced[:8])

assert len(replaced)==N_LAYERS*len(ADALORA_TARGETS),(
    len(replaced),N_LAYERS*len(ADALORA_TARGETS)
)

rows,active,maxrank,rank_stats=adalora_rank_summary(backbone)
print("Initial active rank components:",active,"/",maxrank,"| rank stats:",rank_stats)
print("Expected final rank budget:",len(replaced)*ADALORA_R_TARGET)

adalora_trainable=sum(
    p.numel() for p in backbone.parameters() if p.requires_grad
)
total_backbone=sum(p.numel() for p in backbone.parameters())
print("Backbone total params:",f"{total_backbone/1e6:.3f}M")
print("AdaLoRA trainable params:",f"{adalora_trainable/1e6:.3f}M")
print("Trainable %:",f"{100*adalora_trainable/total_backbone:.4f}%")


Injected AdaLoRA modules: 102
First 8: ['blocks.0.attn.q_proj', 'blocks.0.attn.v_proj', 'blocks.0.attn.o_proj', 'blocks.0.ff.gate_proj', 'blocks.0.ff.up_proj', 'blocks.0.ff.down_proj', 'blocks.1.attn.q_proj', 'blocks.1.attn.v_proj']
Initial active rank components: 1632 / 1632 | rank stats: {'min': 16, 'max': 16, 'mean': 16.0}
Expected final rank budget: 816
Backbone total params: 102.410M
AdaLoRA trainable params: 3.100M
Trainable %: 3.0273%


In [7]:
# ============================================================
# CELL 7 — Download Phase-3 raw JSON files (bypass datasets CastError)
# ============================================================

api = HfApi()
repo_files = api.list_repo_files(
    repo_id=HF_REPO,
    repo_type="dataset",
    token=HF_TOKEN,
)

def exact_repo_file(filename):
    matches = [
        f for f in repo_files
        if os.path.basename(f).lower() == filename.lower()
    ]
    assert len(matches) == 1, (filename, matches)
    return matches[0]

FILES = {
    "CL_train": exact_repo_file("CL_train.json"),
    "CL_dev": exact_repo_file("CL_dev.json"),
    "IT_train": exact_repo_file("IT_train.json"),
    "IT_dev": exact_repo_file("IT_dev.json"),
    "CL_test": exact_repo_file("CL_test.json"),
    "IT_test": exact_repo_file("IT_test.json"),
    "label_vocab": exact_repo_file("label_vocab.json"),
}

print(json.dumps(FILES, indent=2))

def load_repo_json(repo_path):
    p = hf_hub_download(
        repo_id=HF_REPO,
        filename=repo_path,
        repo_type="dataset",
        token=HF_TOKEN,
    )
    with open(p, "r", encoding="utf-8") as f:
        return json.load(f)

cl_train_raw = load_repo_json(FILES["CL_train"])
cl_dev_raw   = load_repo_json(FILES["CL_dev"])
it_train_raw = load_repo_json(FILES["IT_train"])
it_dev_raw   = load_repo_json(FILES["IT_dev"])
cl_test_raw  = load_repo_json(FILES["CL_test"])
it_test_raw  = load_repo_json(FILES["IT_test"])
label_vocab_raw = load_repo_json(FILES["label_vocab"])

assert isinstance(cl_train_raw, list) and isinstance(it_train_raw, list)
assert isinstance(cl_dev_raw, list) and isinstance(it_dev_raw, list)
assert isinstance(cl_test_raw, dict) and isinstance(it_test_raw, dict)

print("CL train/dev:", len(cl_train_raw), len(cl_dev_raw))
print("IT train/dev:", len(it_train_raw), len(it_dev_raw))
print("CL/IT test docs:", len(cl_test_raw), len(it_test_raw))

{
  "CL_train": "Phase_3_RR/Train/CL_train.json",
  "CL_dev": "Phase_3_RR/Train/CL_dev.json",
  "IT_train": "Phase_3_RR/Train/IT_train.json",
  "IT_dev": "Phase_3_RR/Train/IT_dev.json",
  "CL_test": "Phase_3_RR/Test/CL_test.json",
  "IT_test": "Phase_3_RR/Test/IT_test.json",
  "label_vocab": "Phase_3_RR/Train/label_vocab.json"
}


CL_train.json:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

CL_dev.json:   0%|          | 0.00/336k [00:00<?, ?B/s]

IT_train.json:   0%|          | 0.00/1.51M [00:00<?, ?B/s]

IT_dev.json:   0%|          | 0.00/165k [00:00<?, ?B/s]

CL_test.json:   0%|          | 0.00/281k [00:00<?, ?B/s]

IT_test.json:   0%|          | 0.00/194k [00:00<?, ?B/s]

label_vocab.json:   0%|          | 0.00/340 [00:00<?, ?B/s]

CL train/dev: 40 5
IT train/dev: 40 5
CL/IT test docs: 5 5


In [8]:
# ============================================================
# CELL 8 — Labels, examples, class weights, and data checks
# ============================================================

def parse_label_vocab(data):
    if isinstance(data, list):
        labels = list(data)
        label_to_id = {x:i for i,x in enumerate(labels)}
    elif isinstance(data, dict):
        vals = list(data.values())
        if all(isinstance(v, int) and not isinstance(v, bool) for v in vals):
            label_to_id = {str(k): int(v) for k, v in data.items()}
            assert sorted(label_to_id.values()) == list(range(len(label_to_id)))
            labels = [None] * len(label_to_id)
            for lab, idx in label_to_id.items():
                labels[idx] = lab
        elif all(isinstance(v, str) for v in vals):
            tmp = {int(k): v for k, v in data.items()}
            labels = [tmp[i] for i in sorted(tmp)]
            label_to_id = {x:i for i,x in enumerate(labels)}
        else:
            raise ValueError("Unsupported label_vocab format")
    else:
        raise ValueError("Unsupported label_vocab format")

    return labels, label_to_id

LABELS, LABEL_TO_ID = parse_label_vocab(label_vocab_raw)
ID_TO_LABEL = {v:k for k,v in LABEL_TO_ID.items()}
NUM_LABELS = len(LABELS)

assert NUM_LABELS == 13
print("Labels:")
for i, x in enumerate(LABELS):
    print(i, x)

def normalize_id(doc_id):
    doc_id = str(doc_id)
    return doc_id[:-4] if doc_id.endswith(".txt") else doc_id

def validate_labeled_docs(docs, name):
    ids = set()
    n_sent = 0
    for d in docs:
        did = normalize_id(d["id"])
        assert did not in ids, f"{name}: duplicate ID {did}"
        ids.add(did)
        assert isinstance(d["text"], list)
        assert isinstance(d["labels"], list)
        assert len(d["text"]) == len(d["labels"]), did
        assert all(x in LABEL_TO_ID for x in d["labels"]), did
        n_sent += len(d["text"])
    print(f"{name}: docs={len(docs)} sentences={n_sent}")
    return ids

cltr_ids = validate_labeled_docs(cl_train_raw, "CL_train")
cldv_ids = validate_labeled_docs(cl_dev_raw, "CL_dev")
ittr_ids = validate_labeled_docs(it_train_raw, "IT_train")
itdv_ids = validate_labeled_docs(it_dev_raw, "IT_dev")

assert not (cltr_ids & cldv_ids)
assert not (ittr_ids & itdv_ids)
assert not (cldv_ids & itdv_ids)

def position_features(i, n):
    if not USE_POS:
        return [0.0, 0.0, 0.0, 0.0]
    return [i / max(1, n - 1), math.log1p(n) / 8.0, float(i == 0), float(i == n - 1)]

# Flatten into sentence-level records while preserving document context.
def make_labeled_records(docs, domain):
    out = []
    for d in docs:
        sents = d["text"]
        labs = d["labels"]
        did = normalize_id(d["id"])
        for i, (sent, lab) in enumerate(zip(sents, labs)):
            out.append({
                "doc_id": did,
                "sent_idx": i,
                "domain": domain,
                "prev": " ".join(sents[max(0, i-CONTEXT_SENTS):i]) if CONTEXT_SENTS > 0 else "",
                "target": sent,
                "next": " ".join(sents[i+1:i+1+CONTEXT_SENTS]) if CONTEXT_SENTS > 0 else "",
                "pos": position_features(i, len(sents)),
                "label": lab,
                "label_id": LABEL_TO_ID[lab],
            })
    return out

train_records = make_labeled_records(cl_train_raw, "CL") + make_labeled_records(it_train_raw, "IT")
dev_records   = make_labeled_records(cl_dev_raw, "CL") + make_labeled_records(it_dev_raw, "IT")

print("Train sentence records:", len(train_records))
print("Dev sentence records  :", len(dev_records))

train_counts = Counter(x["label"] for x in train_records)
print("\nTrain distribution:")
for lab in LABELS:
    print(f"{lab:30s} {train_counts[lab]:6d}")

# sqrt inverse-frequency; normalize mean to 1; then cap extremes.
N = len(train_records)
C = NUM_LABELS
raw_w = np.array(
    [math.sqrt(N / (C * max(train_counts[lab], 1))) for lab in LABELS],
    dtype=np.float32
)
raw_w = raw_w / raw_w.mean()
class_w = np.clip(raw_w, CLASS_WEIGHT_MIN, CLASS_WEIGHT_MAX)
class_w = torch.tensor(class_w, dtype=torch.float32, device=DEVICE)

print("\nClass weights:")
for lab, w in zip(LABELS, class_w.detach().cpu().tolist()):
    print(f"{lab:30s} {w:.4f}")

Labels:
0 Fact
1 Issue
2 ArgumentPetitioner
3 ArgumentRespondent
4 Statute
5 Dissent
6 PrecedentReliedUpon
7 PrecedentNotReliedUpon
8 PrecedentOverruled
9 RulingByLowerCourt
10 RatioOfTheDecision
11 RulingByPresentCourt
12 None
CL_train: docs=40 sentences=10593
CL_dev: docs=5 sentences=1396
IT_train: docs=40 sentences=6252
IT_dev: docs=5 sentences=746
Train sentence records: 16845
Dev sentence records  : 2142

Train distribution:
Fact                             4098
Issue                             107
ArgumentPetitioner               1613
ArgumentRespondent               1348
Statute                          1099
Dissent                           284
PrecedentReliedUpon              1945
PrecedentNotReliedUpon            569
PrecedentOverruled                  3
RulingByLowerCourt                874
RatioOfTheDecision               4301
RulingByPresentCourt              307
None                              297

Class weights:
Fact                           0.3500
Issue             

In [9]:
# ============================================================
# CELL 9 — Context construction and tokenization
# ============================================================

_ENC_CACHE = {}
def enc(text):
    hit = _ENC_CACHE.get(text)
    if hit is None:
        hit = tokenizer.encode(text)
        _ENC_CACHE[text] = hit
    return hit

def crop_head_tail(ids, budget):
    if len(ids) <= budget:
        return ids
    if budget <= 0:
        return []
    left = budget // 2
    right = budget - left
    return ids[:left] + ids[-right:]

def build_context_ids(domain, prev_text, target_text, next_text, max_len=MAX_LEN):
    # Ordinary text markers only: no tokenizer-vocabulary modification.
    prefix = enc(f"Domain: {domain}\n")
    pmark  = enc("Previous sentence:\n")
    tmark  = enc("\nTarget sentence:\n")
    nmark  = enc("\nNext sentence:\n")
    rmark  = enc("\nRhetorical role:")

    prev_ids = enc(prev_text) if prev_text else []
    targ_ids = enc(target_text)
    next_ids = enc(next_text) if next_text else []

    # Initial caps protect the target sentence and preserve both neighboring sides.
    prev_ids = prev_ids[-PREV_BUDGET:]
    targ_ids = crop_head_tail(targ_ids, TARGET_BUDGET)
    next_ids = next_ids[:NEXT_BUDGET]

    fixed = prefix + pmark + tmark + nmark + rmark
    remaining = max_len - len(fixed)

    if remaining < 8:
        raise RuntimeError("Marker text unexpectedly consumes MAX_LEN.")

    # Fit segments to remaining space, with target priority.
    t_keep = min(len(targ_ids), max(8, min(TARGET_BUDGET, remaining)))
    remaining2 = remaining - t_keep

    p_keep = min(len(prev_ids), remaining2 // 2)
    n_keep = min(len(next_ids), remaining2 - p_keep)

    # Give any spare capacity back to previous/next, then target if available.
    spare = remaining2 - p_keep - n_keep
    if spare > 0:
        add_p = min(spare, len(prev_ids) - p_keep)
        p_keep += add_p
        spare -= add_p
    if spare > 0:
        add_n = min(spare, len(next_ids) - n_keep)
        n_keep += add_n
        spare -= add_n

    prev_ids = prev_ids[-p_keep:] if p_keep else []
    targ_ids = crop_head_tail(targ_ids, t_keep)
    next_ids = next_ids[:n_keep] if n_keep else []

    ids = (
        prefix
        + pmark + prev_ids
        + tmark + targ_ids
        + nmark + next_ids
        + rmark
    )

    if len(ids) > max_len:
        # Absolute safety fallback; preserve the terminal role marker.
        ids = ids[-max_len:]

    assert 1 <= len(ids) <= max_len
    return ids

# Token-length audit
lengths = [
    len(build_context_ids(
        r["domain"], r["prev"], r["target"], r["next"]
    ))
    for r in (train_records[:1000] + dev_records[:500])
]

print("Context-token sample:")
print(" mean:", round(float(np.mean(lengths)), 2))
print(" p95 :", round(float(np.percentile(lengths, 95)), 2))
print(" max :", max(lengths))

Context-token sample:
 mean: 317.94
 p95 : 404.0
 max : 594


In [10]:
# ============================================================
# CELL 10 — Dataset, collator, RR model (Model4 + AdaLoRA)
# ============================================================

class RRDataset(Dataset):
    def __init__(self, records, labeled=True):
        self.records = records
        self.labeled = labeled

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        r = self.records[idx]
        ids = build_context_ids(
            r["domain"], r["prev"], r["target"], r["next"]
        )
        item = {
            "input_ids": ids,
            "doc_id": r["doc_id"],
            "sent_idx": r["sent_idx"],
            "domain": r["domain"],
            "pos": r["pos"],
        }
        if self.labeled:
            item["label_id"] = r["label_id"]
        return item


def collate_rr(batch):
    max_t = max(len(x["input_ids"]) for x in batch)
    B = len(batch)

    input_ids = torch.full(
        (B, max_t),
        tokenizer.pad_token_id,
        dtype=torch.long
    )
    key_valid = torch.zeros((B, max_t), dtype=torch.bool)
    lengths = torch.zeros(B, dtype=torch.long)
    pos = torch.tensor([x["pos"] for x in batch], dtype=torch.float32)

    for i, x in enumerate(batch):
        ids = torch.tensor(x["input_ids"], dtype=torch.long)
        L = len(ids)
        input_ids[i, :L] = ids
        key_valid[i, :L] = True
        lengths[i] = L

    out = {
        "input_ids": input_ids,
        "key_valid": key_valid,
        "lengths": lengths,
        "pos": pos,
        "doc_id": [x["doc_id"] for x in batch],
        "sent_idx": [x["sent_idx"] for x in batch],
        "domain": [x["domain"] for x in batch],
    }

    if "label_id" in batch[0]:
        out["labels"] = torch.tensor(
            [x["label_id"] for x in batch],
            dtype=torch.long
        )

    return out


HEAD_PREFIXES = ("head_norm.", "pos_mlp.", "fc1.", "classifier.")

class RRClassifier(nn.Module):
    def __init__(self, backbone, num_labels=13, dropout=0.15):
        super().__init__()
        self.backbone = backbone
        self.head_norm = RMSNorm(2 * D_MODEL, RMS_EPS)
        self.pos_mlp = nn.Sequential(nn.Linear(4, 64), nn.GELU())
        self.head_drop = nn.Dropout(dropout)
        self.fc1 = nn.Linear(2 * D_MODEL + 64, 512)
        self.classifier = nn.Linear(512, num_labels)

    def forward(self, input_ids, key_valid, lengths, pos):
        h, _ = self.backbone.forward_hidden(
            input_ids, key_valid=key_valid, use_cache=False
        )
        h = h.float()
        row = torch.arange(h.size(0), device=h.device)
        last_h = h[row, lengths - 1]                      # sees whole context (causal) after the role marker
        m = key_valid.unsqueeze(-1).to(h.dtype)
        mean_h = (h * m).sum(1) / m.sum(1).clamp(min=1)
        x = self.head_norm(torch.cat([last_h, mean_h], dim=-1))
        z = torch.cat([x, self.pos_mlp(pos.float())], dim=-1)
        z = F.gelu(self.fc1(self.head_drop(z)))
        return self.classifier(self.head_drop(z))


model = RRClassifier(backbone, NUM_LABELS, HEAD_DROPOUT).to(DEVICE)

# Classifier is trainable; backbone base is frozen except LoRA A/B.
for n_, p in model.named_parameters():
    if n_.startswith(HEAD_PREFIXES):
        p.requires_grad = True

trainable = [(n, p.numel()) for n, p in model.named_parameters() if p.requires_grad]
n_train = sum(x[1] for x in trainable)
n_total = sum(p.numel() for p in model.parameters())

print("Total params    :", f"{n_total/1e6:.3f}M")
print("Trainable params:", f"{n_train/1e6:.3f}M")
print("Trainable %     :", f"{100*n_train/n_total:.4f}%")
print("Trainable tensors:", len(trainable))
print("First trainable names:")
for x in trainable[:12]:
    print(" ", x)

Total params    : 103.107M
Trainable params: 3.797M
Trainable %     : 3.6828%
Trainable tensors: 313
First trainable names:
  ('backbone.blocks.0.attn.q_proj.adalora_A', 10240)
  ('backbone.blocks.0.attn.q_proj.adalora_B', 10240)
  ('backbone.blocks.0.attn.q_proj.adalora_e', 16)
  ('backbone.blocks.0.attn.v_proj.adalora_A', 10240)
  ('backbone.blocks.0.attn.v_proj.adalora_B', 2048)
  ('backbone.blocks.0.attn.v_proj.adalora_e', 16)
  ('backbone.blocks.0.attn.o_proj.adalora_A', 10240)
  ('backbone.blocks.0.attn.o_proj.adalora_B', 10240)
  ('backbone.blocks.0.attn.o_proj.adalora_e', 16)
  ('backbone.blocks.0.ff.gate_proj.adalora_A', 10240)
  ('backbone.blocks.0.ff.gate_proj.adalora_B', 32768)
  ('backbone.blocks.0.ff.gate_proj.adalora_e', 16)


In [11]:
# ============================================================
# CELL 11 — Dataloaders, optimizer, scheduler
# ============================================================
class SortishBatchSampler(torch.utils.data.Sampler):
    """Deterministic shuffled length buckets to reduce padding."""
    def __init__(self,lengths,batch_size,seed=SEED,bucket_mult=LENGTH_BUCKET_MULT):
        self.lengths=list(map(int,lengths))
        self.batch_size=int(batch_size)
        self.seed=int(seed)
        self.bucket_size=max(self.batch_size,self.batch_size*int(bucket_mult))
        self.epoch=0
    def set_epoch(self,epoch): self.epoch=int(epoch)
    def __len__(self): return math.ceil(len(self.lengths)/self.batch_size)
    def __iter__(self):
        rng=np.random.default_rng(self.seed+self.epoch)
        idx=np.arange(len(self.lengths),dtype=np.int64)
        rng.shuffle(idx)
        batches=[]
        for s in range(0,len(idx),self.bucket_size):
            chunk=idx[s:s+self.bucket_size]
            chunk=sorted(
                chunk.tolist(),
                key=lambda i:self.lengths[i],
                reverse=True,
            )
            batches.extend(
                chunk[j:j+self.batch_size]
                for j in range(0,len(chunk),self.batch_size)
            )
        rng.shuffle(batches)
        yield from batches


# Cache token lengths once for grouping.
_train_ds=RRDataset(train_records,labeled=True)
_dev_ds=RRDataset(dev_records,labeled=True)

train_lengths=[
    len(build_context_ids(r["domain"],r["prev"],r["target"],r["next"]))
    for r in train_records
]

if LENGTH_GROUPED_TRAINING:
    train_loader=DataLoader(
        _train_ds,
        batch_sampler=SortishBatchSampler(
            train_lengths,MICRO_BATCH_SIZE,seed=SEED
        ),
        collate_fn=collate_rr,
        num_workers=NUM_WORKERS,
        pin_memory=(DEVICE.type=="cuda"),
        persistent_workers=NUM_WORKERS>0,
    )
else:
    g=torch.Generator(); g.manual_seed(SEED)
    train_loader=DataLoader(
        _train_ds,batch_size=MICRO_BATCH_SIZE,shuffle=True,
        generator=g,collate_fn=collate_rr,
        num_workers=NUM_WORKERS,pin_memory=(DEVICE.type=="cuda"),
        persistent_workers=NUM_WORKERS>0,
    )

dev_loader=DataLoader(
    _dev_ds,batch_size=EVAL_BATCH_SIZE,shuffle=False,
    collate_fn=collate_rr,num_workers=NUM_WORKERS,
    pin_memory=(DEVICE.type=="cuda"),
    persistent_workers=NUM_WORKERS>0,
)

adalora_params=[]
head_params=[]
for name,p in model.named_parameters():
    if not p.requires_grad:
        continue
    if "adalora_" in name:
        adalora_params.append(p)
    else:
        assert name.startswith(HEAD_PREFIXES),name
        head_params.append(p)

def make_adamw(adapter_params,head_params):
    groups=[
        {"params":adapter_params,"lr":ADALORA_LR,"weight_decay":0.0},
        {"params":head_params,"lr":HEAD_LR,"weight_decay":WEIGHT_DECAY},
    ]
    if DEVICE.type=="cuda":
        try:
            opt=torch.optim.AdamW(groups,fused=True)
            print("AdamW fused: True")
            return opt
        except Exception as e:
            print("Fused AdamW unavailable:",repr(e)[:120])
    print("AdamW fused: False")
    return torch.optim.AdamW(groups)

optimizer=make_adamw(adalora_params,head_params)

updates_per_epoch=math.ceil(len(train_loader)/GRAD_ACCUM_STEPS)
total_updates=NUM_EPOCHS*updates_per_epoch
warmup_updates=max(1,int(total_updates*WARMUP_RATIO))

def lr_mult(step):
    if step<warmup_updates:
        return max(1e-8,step/warmup_updates)
    p=(step-warmup_updates)/max(1,total_updates-warmup_updates)
    return 0.5*(1+math.cos(math.pi*min(max(p,0.0),1.0)))

scheduler=torch.optim.lr_scheduler.LambdaLR(optimizer,lr_lambda=lr_mult)
criterion=nn.CrossEntropyLoss(weight=class_w,label_smoothing=LABEL_SMOOTHING)

amp_enabled=bool(USE_AMP and DEVICE.type=="cuda")
scaler=torch.amp.GradScaler("cuda",enabled=amp_enabled)

print("Train batches/epoch:",len(train_loader))
print("Updates/epoch:",updates_per_epoch)
print("Total updates:",total_updates)
print("Warmup:",warmup_updates)
print("AdaLoRA initial global rank budget:",len(replaced)*ADALORA_R_INIT)
print("AdaLoRA final global rank budget  :",len(replaced)*ADALORA_R_TARGET)


AdamW fused: True
Train batches/epoch: 2106
Updates/epoch: 1053
Total updates: 3159
Warmup: 189
AdaLoRA initial global rank budget: 1632
AdaLoRA final global rank budget  : 816


In [12]:
# ============================================================
# CELL 12 — Metrics and evaluation
# ============================================================

@torch.no_grad()
def predict_loader(model, loader):
    model.eval()

    all_true = []
    all_pred = []
    all_meta = []

    for batch in loader:
        input_ids = batch["input_ids"].to(DEVICE, non_blocking=True)
        key_valid = batch["key_valid"].to(DEVICE, non_blocking=True)
        lengths = batch["lengths"].to(DEVICE, non_blocking=True)
        pos = batch["pos"].to(DEVICE, non_blocking=True)

        with torch.amp.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=amp_enabled
        ):
            logits = model(input_ids, key_valid, lengths, pos)

        pred = logits.float().argmax(-1).cpu().tolist()
        all_pred.extend(pred)

        if "labels" in batch:
            all_true.extend(batch["labels"].tolist())

        all_meta.extend(zip(
            batch["doc_id"],
            batch["sent_idx"],
            batch["domain"]
        ))

    return all_true, all_pred, all_meta


def score_ids(y_true, y_pred, title="Combined"):
    label_ids = list(range(NUM_LABELS))

    macro_f1 = f1_score(
        y_true, y_pred,
        labels=label_ids,
        average="macro",
        zero_division=0
    )
    macro_p = precision_score(
        y_true, y_pred,
        labels=label_ids,
        average="macro",
        zero_division=0
    )
    macro_r = recall_score(
        y_true, y_pred,
        labels=label_ids,
        average="macro",
        zero_division=0
    )

    print(f"{title}: F1={macro_f1:.6f} P={macro_p:.6f} R={macro_r:.6f}")
    return macro_f1, macro_p, macro_r



def dev_label_ceiling_diagnostic(y_true):
    present = sorted(set(y_true))
    absent = [i for i in range(NUM_LABELS) if i not in present]
    ceiling = len(present) / NUM_LABELS
    print(
        f"Dev contains {len(present)}/{NUM_LABELS} labels; "
        f"formal Macro-F1 ceiling from label absence = {ceiling:.6f}"
    )
    if absent:
        print("Absent dev labels:", [LABELS[i] for i in absent])
    return ceiling


def detailed_dev_eval(model, verbose=True):
    y_true, y_pred, meta = predict_loader(model, dev_loader)

    combined = score_ids(y_true, y_pred, "Combined CL+IT")
    dev_label_ceiling_diagnostic(y_true)

    # ---- diagnostics: collapse check + F1 over labels actually present in dev
    pred_c, gold_c = Counter(y_pred), Counter(y_true)
    present = sorted(set(y_true))
    f1_present = f1_score(y_true, y_pred, labels=present, average="macro", zero_division=0)
    print(f"   Macro-F1 over {len(present)} labels present in dev: {f1_present:.6f}")
    print("   gold counts:", {LABELS[k]: gold_c[k] for k in present})
    print("   pred counts:", {LABELS[k]: pred_c[k] for k in sorted(pred_c)})
    if max(pred_c.values()) / max(1, len(y_pred)) > 0.60:
        print("   ⚠️ >60% of predictions are one class — model may be collapsing.")

    for dom in ["CL", "IT"]:
        idx = [i for i, x in enumerate(meta) if x[2] == dom]
        yt = [y_true[i] for i in idx]
        yp = [y_pred[i] for i in idx]
        score_ids(yt, yp, dom)

    if verbose:
        print("\nPer-label report (all 13 labels explicitly included):")
        print(classification_report(
            y_true, y_pred,
            labels=list(range(NUM_LABELS)),
            target_names=LABELS,
            digits=4,
            zero_division=0
        ))

    return combined, y_true, y_pred, meta

@torch.no_grad()
def predict_logits(model, loader):
    """Returns (logits[N,13] float32, meta list, y_true list) in loader order."""
    model.eval()
    outs, meta, y_true = [], [], []
    for batch in loader:
        input_ids = batch["input_ids"].to(DEVICE, non_blocking=True)
        key_valid = batch["key_valid"].to(DEVICE, non_blocking=True)
        lengths = batch["lengths"].to(DEVICE, non_blocking=True)
        pos = batch["pos"].to(DEVICE, non_blocking=True)
        with torch.amp.autocast(device_type="cuda", dtype=torch.float16, enabled=amp_enabled):
            lg = model(input_ids, key_valid, lengths, pos)
        outs.append(lg.float().cpu().numpy())
        meta.extend(zip(batch["doc_id"], batch["sent_idx"], batch["domain"]))
        if "labels" in batch:
            y_true.extend(batch["labels"].tolist())
    return np.concatenate(outs, 0), meta, y_true


In [13]:
# ============================================================
# CELL 13 — Train AdaLoRA; adaptive rank allocation + mid-epoch dev selection
# ============================================================
def fmt_sec(s):
    s=max(0,int(s)); h,s=divmod(s,3600); m,s=divmod(s,60)
    return f"{h:02d}:{m:02d}:{s:02d}"

best_f1=-1.0
best_epoch=None
best_frac=None
best_batch_in_epoch=None
best_global_update=None
bad_evals=0
global_update=0
history=[]
rank_history=[]
train_start=time.time()

optimizer.zero_grad(set_to_none=True)

for epoch in range(1,NUM_EPOCHS+1):
    if hasattr(train_loader.batch_sampler,"set_epoch"):
        train_loader.batch_sampler.set_epoch(epoch)

    model.train()
    running=torch.zeros((),device=DEVICE,dtype=torch.float32)
    seen_batches=0
    ep0=time.time()

    eval_steps=sorted(set(
        min(len(train_loader),max(GRAD_ACCUM_STEPS,int(round(len(train_loader)*j/EVAL_PER_EPOCH/GRAD_ACCUM_STEPS))*GRAD_ACCUM_STEPS))
        for j in range(1,EVAL_PER_EPOCH+1)
    ))
    eval_steps[-1]=len(train_loader)
    if epoch==1:
        print("Dev evaluation at batches:",eval_steps,"of",len(train_loader))

    stop_training=False

    for step,batch in enumerate(train_loader,1):
        input_ids=batch["input_ids"].to(DEVICE,non_blocking=True)
        key_valid=batch["key_valid"].to(DEVICE,non_blocking=True)
        lengths=batch["lengths"].to(DEVICE,non_blocking=True)
        pos=batch["pos"].to(DEVICE,non_blocking=True)
        labels=batch["labels"].to(DEVICE,non_blocking=True)

        with torch.amp.autocast(
            device_type="cuda",dtype=torch.float16,enabled=amp_enabled
        ):
            logits=model(input_ids,key_valid,lengths,pos)
            loss=criterion(logits,labels)
            scaled_loss=loss/GRAD_ACCUM_STEPS

        if step%NAN_CHECK_EVERY==0 or step==len(train_loader):
            if not bool(torch.isfinite(loss.detach()).item()):
                raise FloatingPointError(
                    f"Non-finite loss epoch={epoch} batch={step}"
                )

        scaler.scale(scaled_loss).backward()
        running.add_(loss.detach().float())
        seen_batches+=1

        do_update=(step%GRAD_ACCUM_STEPS==0 or step==len(train_loader))
        if do_update:
            scaler.unscale_(optimizer)

            # Update importance BEFORE optimizer clears gradients.
            adalora_update_importance(model)

            torch.nn.utils.clip_grad_norm_(
                [p for p in model.parameters() if p.requires_grad],
                GRAD_CLIP
            )

            old_scale=scaler.get_scale()
            scaler.step(optimizer)
            scaler.update()
            new_scale=scaler.get_scale()
            optimizer.zero_grad(set_to_none=True)

            if new_scale>=old_scale:
                scheduler.step()
                global_update+=1

                # Periodic global rank allocation.
                if (
                    global_update%ADALORA_PRUNE_INTERVAL==0
                    or global_update==int(total_updates*ADALORA_TFINAL_FRAC)
                ):
                    target_budget=rank_budget_for_update(
                        model,global_update,total_updates
                    )
                    pruned=adalora_prune_to_budget(
                        model,target_budget
                    )
                    _,active,maxrank,rank_stats=adalora_rank_summary(model)
                    rank_history.append({
                        "update":int(global_update),
                        "target_budget":int(target_budget),
                        "active":int(active),
                        "maximum":int(maxrank),
                        "rank_min":int(rank_stats["min"]),
                        "rank_mean":float(rank_stats["mean"]),
                        "rank_max":int(rank_stats["max"]),
                        "pruned_now":int(pruned),
                    })

        if step%PRINT_EVERY==0 or step==len(train_loader):
            elapsed=time.time()-ep0
            rate=elapsed/max(step,1)
            eta_ep=rate*(len(train_loader)-step)
            completed=(epoch-1)+step/len(train_loader)
            eta_total=rate*len(train_loader)*max(0,NUM_EPOCHS-completed)
            _,active,maxrank,rank_stats=adalora_rank_summary(model)

            print(
                f"epoch={epoch} batch={step}/{len(train_loader)} "
                f"loss={float((running/max(seen_batches,1)).item()):.4f} "
                f"lr_ada={optimizer.param_groups[0]['lr']:.2e} "
                f"lr_head={optimizer.param_groups[1]['lr']:.2e} "
                f"rank={active}/{maxrank} "
                f"(min/mean/max={rank_stats['min']}/{rank_stats['mean']:.1f}/{rank_stats['max']}) "
                f"elapsed={fmt_sec(elapsed)} "
                f"ETA_epoch={fmt_sec(eta_ep)} "
                f"ETA_maxtrain≈{fmt_sec(eta_total)}",
                flush=True
            )

        if step in eval_steps:
            metrics,y_true,y_pred,meta=detailed_dev_eval(model,verbose=False)
            macro_f1,macro_p,macro_r=metrics
            frac=(epoch-1)+step/len(train_loader)
            _,active,maxrank,rank_stats=adalora_rank_summary(model)

            row={
                "epoch":epoch,
                "epoch_frac":float(frac),
                "batch_in_epoch":int(step),
                "global_update":int(global_update),
                "active_rank_components":int(active),
                "max_rank_components":int(maxrank),
                "train_loss":float((running/max(seen_batches,1)).item()),
                "macro_f1":float(macro_f1),
                "macro_precision":float(macro_p),
                "macro_recall":float(macro_r),
                "elapsed_sec":time.time()-ep0,
            }
            history.append(row)
            print("Eval summary:",row)

            if macro_f1>best_f1:
                best_f1=float(macro_f1)
                best_epoch=int(math.ceil(frac))
                best_frac=float(frac)
                best_batch_in_epoch=int(step)
                best_global_update=int(global_update)
                bad_evals=0

                task_state={
                    k:v.detach().cpu()
                    for k,v in model.state_dict().items()
                    if ("adalora_" in k or k.startswith(HEAD_PREFIXES))
                }

                rank_rows,rank_active,rank_max,rank_stats=adalora_rank_summary(model)
                rank_map={n:int(a) for n,a,_ in rank_rows}

                torch.save({
                    "task_state":task_state,
                    "best_epoch":best_epoch,
                    "best_frac":best_frac,
                    "best_batch_in_epoch":best_batch_in_epoch,
                    "best_global_update":best_global_update,
                    "best_macro_f1":best_f1,
                    "history":history,
                    "rank_history":rank_history,
                    "rank_map":rank_map,
                    "rank_stats":rank_stats,
                    "labels":LABELS,
                    "label_to_id":LABEL_TO_ID,
                    "config":{
                        "experiment":"Phase3 Exp5 Model4 AdaLoRA",
                        "r_init":ADALORA_R_INIT,
                        "r_target":ADALORA_R_TARGET,
                        "alpha":ADALORA_ALPHA,
                        "dropout":ADALORA_DROPOUT,
                        "targets":ADALORA_TARGETS,
                        "tinit_frac":ADALORA_TINIT_FRAC,
                        "tfinal_frac":ADALORA_TFINAL_FRAC,
                        "prune_interval":ADALORA_PRUNE_INTERVAL,
                        "max_len":MAX_LEN,
                        "context_sents":CONTEXT_SENTS,
                    }
                },BEST_CKPT)

                print(
                    f"✅ NEW BEST at epoch {frac:.3f} | "
                    f"batch={step} | update={global_update} | "
                    f"rank={rank_active}/{rank_max}: {BEST_CKPT}"
                )
            else:
                bad_evals+=1

            print(
                f"Best so far: epoch_frac={best_frac:.3f} "
                f"Macro-F1={best_f1:.6f} "
                f"| non-improving evals={bad_evals}\n"
            )

            if bad_evals>=EARLY_STOP_PATIENCE_EVALS:
                print(
                    f"⏹️ Early stopping after {bad_evals} "
                    "non-improving dev evaluations."
                )
                stop_training=True
                break

            model.train()

    if stop_training:
        break

with open(ROOT/"history.json","w",encoding="utf-8") as f:
    json.dump(history,f,indent=2)
with open(ROOT/"rank_history.json","w",encoding="utf-8") as f:
    json.dump(rank_history,f,indent=2)

assert best_frac is not None
print("Training complete | elapsed:",fmt_sec(time.time()-train_start))
print(
    "BEST_EPOCH =",best_epoch,
    "| BEST_EPOCH_FRAC =",best_frac,
    "| BEST_BATCH =",best_batch_in_epoch,
    "| BEST_UPDATE =",best_global_update,
    "| BEST_MACRO_F1 =",best_f1
)


Dev evaluation at batches: [526, 1052, 1580, 2106] of 2106
epoch=1 batch=250/2106 loss=2.0853 lr_ada=1.32e-04 lr_head=6.61e-04 rank=1632/1632 (min/mean/max=16/16.0/16) elapsed=00:00:53 ETA_epoch=00:06:38 ETA_maxtrain≈00:21:44
epoch=1 batch=500/2106 loss=1.8577 lr_ada=2.00e-04 lr_head=9.99e-04 rank=1508/1632 (min/mean/max=8/14.8/16) elapsed=00:01:46 ETA_epoch=00:05:42 ETA_maxtrain≈00:20:39
Combined CL+IT: F1=0.330773 P=0.398676 R=0.331833
Dev contains 11/13 labels; formal Macro-F1 ceiling from label absence = 0.846154
Absent dev labels: ['Dissent', 'PrecedentOverruled']
   Macro-F1 over 11 labels present in dev: 0.390913
   gold counts: {'Fact': 444, 'Issue': 15, 'ArgumentPetitioner': 400, 'ArgumentRespondent': 272, 'Statute': 120, 'PrecedentReliedUpon': 254, 'PrecedentNotReliedUpon': 50, 'RulingByLowerCourt': 102, 'RatioOfTheDecision': 403, 'RulingByPresentCourt': 49, 'None': 33}
   pred counts: {'Fact': 429, 'Issue': 2, 'ArgumentPetitioner': 369, 'ArgumentRespondent': 56, 'Statute': 1

In [14]:
# ============================================================
# CELL 14 — Reload exact best AdaLoRA checkpoint + create DEV prediction/evaluator files
# ============================================================
best_obj=torch.load(BEST_CKPT,map_location="cpu",weights_only=False)
missing,unexpected=model.load_state_dict(best_obj["task_state"],strict=False)

bad_missing=[
    k for k in missing
    if ("adalora_" in k or k.startswith(HEAD_PREFIXES))
]
assert not bad_missing,bad_missing
assert not unexpected,unexpected

best_f1=float(best_obj["best_macro_f1"])
best_epoch=int(best_obj["best_epoch"])
best_frac=float(best_obj.get("best_frac",best_epoch))
print(
    "Reloaded best AdaLoRA checkpoint:",
    "epoch_frac=",best_frac,
    "Macro-F1=",best_f1
)

# Full dev logits are saved for reproducibility / later ensemble analysis.
dev_logits,dev_meta,dev_y=predict_logits(model,dev_loader)
np.savez_compressed(
    ROOT/"dev_logits.npz",
    logits=dev_logits,
    y_true=np.asarray(dev_y,dtype=np.int64),
    doc_id=np.asarray([x[0] for x in dev_meta],dtype=object),
    sent_idx=np.asarray([x[1] for x in dev_meta],dtype=np.int64),
    domain=np.asarray([x[2] for x in dev_meta],dtype=object),
)

def records_to_prediction(meta,pred_ids):
    by_doc={}
    for (did,si,dom),pid in zip(meta,pred_ids):
        by_doc.setdefault(did,[]).append((si,LABELS[int(pid)]))
    out={}
    for did,items in by_doc.items():
        items.sort(key=lambda x:x[0])
        out[did]=[lab for _,lab in items]
    return out

dev_pred=records_to_prediction(dev_meta,dev_logits.argmax(-1))
with open(DEV_PRED,"w",encoding="utf-8") as f:
    json.dump(dev_pred,f,ensure_ascii=False,indent=2)

def write_truth(docs,path):
    truth={}
    for d in docs:
        did=normalize_id(d["id"])
        truth[did]=list(d["labels"])
    with open(path,"w",encoding="utf-8") as f:
        json.dump(truth,f,ensure_ascii=False,indent=2)

write_truth(cl_dev_raw,CL_DEV_TRUTH)
write_truth(it_dev_raw,IT_DEV_TRUTH)
with open(LABELS_JSON,"w",encoding="utf-8") as f:
    json.dump(LABELS,f,ensure_ascii=False,indent=2)

for p in [CL_DEV_TRUTH,IT_DEV_TRUTH,LABELS_JSON,DEV_PRED]:
    assert Path(p).is_file(),p

print("✅ DEV prediction/truth/labels written.")


Reloaded best AdaLoRA checkpoint: epoch_frac= 2.2497625830959165 Macro-F1= 0.4228304375350831
✅ DEV prediction/truth/labels written.


In [15]:
# ============================================================
# CELL 15 — EXACT local Phase-3 evaluator script supplied by course
# ============================================================

EVALUATOR_CODE = r'''#!/usr/bin/env python3
"""Local evaluator for Phase 3.

The competition scorer evaluates two gold/reference JSON files together and
one participant prediction.json. This local evaluator mirrors that setup
using the student's own held-out splits.
"""
import argparse
import json
import sys

from sklearn.metrics import f1_score, precision_score, recall_score

EXPECTED_LABEL_COUNT = 13


def load_json(path):
    with open(path, encoding="utf-8") as f:
        return json.load(f)


def normalize_id(doc_id):
    if not isinstance(doc_id, str) or not doc_id.strip():
        raise ValueError("Document IDs must be non-empty strings.")
    return doc_id[:-4] if doc_id.endswith(".txt") else doc_id


def load_vocab(data):
    if isinstance(data, list):
        labels = data
    elif isinstance(data, dict) and data:
        values = list(data.values())
        if all(isinstance(v, int) and not isinstance(v, bool) for v in values):
            labels = list(data.keys())
        elif all(isinstance(v, str) for v in values):
            labels = values
        else:
            raise ValueError("Unsupported labels.json format.")
    else:
        raise ValueError("labels.json must be a non-empty list or object.")

    if len(labels) != EXPECTED_LABEL_COUNT:
        raise ValueError(
            f"Expected {EXPECTED_LABEL_COUNT} labels, found {len(labels)}."
        )
    if any(not isinstance(label, str) or not label.strip() for label in labels):
        raise ValueError("Labels must be unique non-empty strings.")
    if len(set(labels)) != len(labels):
        raise ValueError("Labels must be unique non-empty strings.")
    return labels


def normalize_reference(data, source_name):
    if not isinstance(data, dict):
        raise ValueError(f"{source_name}: reference data must be a JSON object.")

    normalized = {}
    for raw_id, labels in data.items():
        doc_id = normalize_id(raw_id)
        if doc_id in normalized:
            raise ValueError(
                f"{source_name}: duplicate document ID after .txt normalization: {doc_id}"
            )
        normalized[doc_id] = labels
    return normalized


def normalize_predictions(data):
    if not isinstance(data, dict):
        raise ValueError("prediction.json must be a JSON object mapping document IDs to label lists.")

    normalized = {}
    for raw_id, labels in data.items():
        doc_id = normalize_id(raw_id)
        if doc_id in normalized:
            raise ValueError(
                f"prediction.json: duplicate document ID after .txt normalization: {doc_id}"
            )
        normalized[doc_id] = labels
    return normalized


def main():
    parser = argparse.ArgumentParser(
        description=(
            "Evaluate Phase 3 sentence-label predictions on two local held-out "
            "reference splits, mirroring the competition's combined scoring setup."
        )
    )
    parser.add_argument("--truth-cl", required=True)
    parser.add_argument("--truth-it", required=True)
    parser.add_argument("--prediction", required=True)
    parser.add_argument("--labels", required=True)
    args = parser.parse_args()

    truth_cl = normalize_reference(load_json(args.truth_cl), "--truth-cl")
    truth_it = normalize_reference(load_json(args.truth_it), "--truth-it")
    predictions = normalize_predictions(load_json(args.prediction))
    labels = load_vocab(load_json(args.labels))
    allowed = set(labels)

    overlap = set(truth_cl) & set(truth_it)
    if overlap:
        sample = sorted(overlap)[:5]
        suffix = "..." if len(overlap) > 5 else ""
        raise ValueError(
            "The two reference splits contain overlapping document IDs: "
            + ", ".join(sample)
            + suffix
        )

    truth = {}
    truth.update(truth_cl)
    truth.update(truth_it)

    if set(truth) != set(predictions):
        missing = sorted(set(truth) - set(predictions))
        extra = sorted(set(predictions) - set(truth))
        parts = [f"missing={len(missing)}", f"extra={len(extra)}"]
        if missing:
            parts.append(f"example missing ID(s)={missing[:5]}")
        if extra:
            parts.append(f"example extra ID(s)={extra[:5]}")
        raise ValueError("ID mismatch: " + "; ".join(parts))

    y_true = []
    y_pred = []

    for doc_id in truth:
        gold = truth[doc_id]
        pred = predictions[doc_id]

        if not isinstance(gold, list) or not isinstance(pred, list):
            raise ValueError(f"{doc_id}: labels must be JSON lists.")
        if len(gold) != len(pred):
            raise ValueError(
                f"{doc_id}: sentence-count mismatch ({len(gold)} vs {len(pred)})."
            )
        if any(label not in allowed for label in gold + pred):
            raise ValueError(f"{doc_id}: unknown label encountered.")

        y_true.extend(gold)
        y_pred.extend(pred)

    score = f1_score(y_true, y_pred, labels=labels, average="macro", zero_division=0)
    macro_precision = precision_score(y_true, y_pred, labels=labels, average="macro", zero_division=0)
    macro_recall = recall_score(y_true, y_pred, labels=labels, average="macro", zero_division=0)

    print(f"CL documents: {len(truth_cl)}")
    print(f"IT documents: {len(truth_it)}")
    print(f"Total documents: {len(truth)}")
    print(f"Total sentences: {len(y_true)}")
    print(f"Macro-F1:        {score:.6f}")
    print(f"Macro-Precision: {macro_precision:.6f}")
    print(f"Macro-Recall:    {macro_recall:.6f}")


if __name__ == "__main__":
    try:
        main()
    except Exception as exc:
        print(f"EVALUATION FAILED: {exc}", file=sys.stderr)
        sys.exit(1)
'''

with open(EVAL_SCRIPT, "w", encoding="utf-8") as f:
    f.write(EVALUATOR_CODE)

print("Saved exact evaluator to:", EVAL_SCRIPT)

Saved exact evaluator to: /kaggle/working/phase3_exp5_model4_adalora/evaluate_phase3.py


In [16]:
# ============================================================
# CELL 16 — RUN THE OFFICIAL-STYLE LOCAL EVALUATOR
# Do not continue to final refit until this passes.
# ============================================================

cmd = [
    sys.executable,
    str(EVAL_SCRIPT),
    "--truth-cl", str(CL_DEV_TRUTH),
    "--truth-it", str(IT_DEV_TRUTH),
    "--prediction", str(DEV_PRED),
    "--labels", str(LABELS_JSON),
]

print("Running:")
print(" ".join(cmd))

result = subprocess.run(
    cmd,
    capture_output=True,
    text=True
)

print("\nSTDOUT:\n", result.stdout)
if result.stderr:
    print("\nSTDERR:\n", result.stderr)

assert result.returncode == 0, "❌ Phase-3 evaluator failed."
print("✅ DEV prediction passed the exact Phase-3 evaluator.")

Running:
/usr/bin/python3 /kaggle/working/phase3_exp5_model4_adalora/evaluate_phase3.py --truth-cl /kaggle/working/phase3_exp5_model4_adalora/cl_dev_truth.json --truth-it /kaggle/working/phase3_exp5_model4_adalora/it_dev_truth.json --prediction /kaggle/working/phase3_exp5_model4_adalora/prediction_dev.json --labels /kaggle/working/phase3_exp5_model4_adalora/labels.json

STDOUT:
 CL documents: 5
IT documents: 5
Total documents: 10
Total sentences: 2142
Macro-F1:        0.422830
Macro-Precision: 0.447636
Macro-Recall:    0.412780

✅ DEV prediction passed the exact Phase-3 evaluator.


In [17]:
# ============================================================
# CELL 16b — Optional transition-aware decoding tuned on DEV
#
# Improvements:
#   * CL and IT transition/prior statistics estimated separately
#   * additive smoothing
#   * no hard "impossible label" bans
#   * only adopt decoding when DEV Macro-F1 improves by >= 0.005
# ============================================================

from sklearn.metrics import f1_score as _f1

def transition_matrix(docs_list, smooth=1.0):
    T = np.full((NUM_LABELS, NUM_LABELS), float(smooth), dtype=np.float64)
    for d in docs_list:
        ids = [LABEL_TO_ID[x] for x in d["labels"]]
        for a, b in zip(ids[:-1], ids[1:]):
            T[a, b] += 1.0
    T /= T.sum(1, keepdims=True)
    return np.log(T)

def class_log_prior(docs_list, smooth=1.0):
    c = np.full(NUM_LABELS, float(smooth), dtype=np.float64)
    for d in docs_list:
        for x in d["labels"]:
            c[LABEL_TO_ID[x]] += 1.0
    return np.log(c / c.sum())

def log_softmax_np(x):
    x = x - x.max(-1, keepdims=True)
    return x - np.log(np.exp(x).sum(-1, keepdims=True))

def viterbi_decode(logp, trans, beta):
    n, K = logp.shape
    score = logp[0].copy()
    back = np.zeros((n, K), dtype=np.int32)
    tr = beta * trans

    for t in range(1, n):
        cand = score[:, None] + tr
        back[t] = cand.argmax(0)
        score = cand[back[t], np.arange(K)] + logp[t]

    path = np.zeros(n, dtype=np.int64)
    path[-1] = int(score.argmax())

    for t in range(n - 1, 0, -1):
        path[t - 1] = back[t, path[t]]

    return path

def build_domain_stats(cl_docs, it_docs):
    return {
        "CL": {
            "trans": transition_matrix(cl_docs, smooth=1.0),
            "prior": class_log_prior(cl_docs, smooth=1.0),
        },
        "IT": {
            "trans": transition_matrix(it_docs, smooth=1.0),
            "prior": class_log_prior(it_docs, smooth=1.0),
        },
    }

def decode_docs(logits, meta, domain_stats, alpha, beta):
    # logits aligned with meta=(doc_id, sent_idx, domain)
    by_doc = {}
    for i, (did, si, dom) in enumerate(meta):
        by_doc.setdefault(did, []).append((si, i, dom))

    pred = np.zeros(len(meta), dtype=np.int64)

    for did, items in by_doc.items():
        items.sort(key=lambda x: x[0])
        idx = [i for _, i, _ in items]
        dom = items[0][2]
        stats = domain_stats[dom]

        lp = (
            log_softmax_np(logits[idx].astype(np.float64))
            - alpha * stats["prior"][None, :]
        )

        pred[idx] = (
            viterbi_decode(lp, stats["trans"], beta)
            if beta > 0
            else lp.argmax(-1)
        )

    return pred

# Tune only on held-out DEV predictions.
dev_logits, dev_meta, dev_y = predict_logits(model, dev_loader)

domain_stats_train = build_domain_stats(
    cl_train_raw,
    it_train_raw,
)

def dev_f1(alpha, beta):
    p = decode_docs(
        dev_logits,
        dev_meta,
        domain_stats_train,
        alpha,
        beta
    )
    return _f1(
        dev_y,
        p,
        labels=list(range(NUM_LABELS)),
        average="macro",
        zero_division=0
    )

rows = []
for alpha in [-0.25, 0.0, 0.25, 0.5, 0.75]:
    for beta in [0.0, 0.25, 0.5, 1.0]:
        rows.append((dev_f1(alpha, beta), alpha, beta))

base_f1 = dev_f1(0.0, 0.0)
rows.sort(key=lambda r: (-r[0], r[2], abs(r[1])))

print(f"Baseline argmax (alpha=0, beta=0): {base_f1:.6f}")
print("Top decoding settings (dev F1, alpha, beta):")
for r in rows[:8]:
    print(f"  {r[0]:.6f}  alpha={r[1]:+.2f}  beta={r[2]:.2f}")

if rows[0][0] - base_f1 >= 0.005:
    DECODE_ALPHA, DECODE_BETA = rows[0][1], rows[0][2]
else:
    DECODE_ALPHA, DECODE_BETA = 0.0, 0.0

print("SELECTED decoding: alpha=", DECODE_ALPHA, "beta=", DECODE_BETA)

dec_ids = decode_docs(
    dev_logits,
    dev_meta,
    domain_stats_train,
    DECODE_ALPHA,
    DECODE_BETA
)

dev_pred_dec = records_to_prediction(dev_meta, dec_ids)
DEV_PRED_DEC = ROOT / "prediction_dev_decoded.json"

with open(DEV_PRED_DEC, "w", encoding="utf-8") as f:
    json.dump(dev_pred_dec, f, ensure_ascii=False, indent=2)

# Run the exact supplied evaluator again on the decoded DEV prediction.
res = subprocess.run(
    [
        sys.executable,
        str(EVAL_SCRIPT),
        "--truth-cl", str(CL_DEV_TRUTH),
        "--truth-it", str(IT_DEV_TRUTH),
        "--prediction", str(DEV_PRED_DEC),
        "--labels", str(LABELS_JSON),
    ],
    capture_output=True,
    text=True
)

print(res.stdout)
if res.stderr:
    print(res.stderr)

assert res.returncode == 0, "Decoded DEV prediction failed the exact evaluator."
print("✅ Decoded DEV prediction passed the exact Phase-3 evaluator.")

Baseline argmax (alpha=0, beta=0): 0.422830
Top decoding settings (dev F1, alpha, beta):
  0.444992  alpha=+0.50  beta=1.00
  0.433823  alpha=+0.25  beta=1.00
  0.433234  alpha=+0.00  beta=0.50
  0.432249  alpha=-0.25  beta=0.25
  0.431804  alpha=+0.00  beta=1.00
  0.429951  alpha=+0.25  beta=0.25
  0.428476  alpha=-0.25  beta=0.50
  0.428218  alpha=+0.00  beta=0.25
SELECTED decoding: alpha= 0.5 beta= 1.0
CL documents: 5
IT documents: 5
Total documents: 10
Total sentences: 2142
Macro-F1:        0.444992
Macro-Precision: 0.495620
Macro-Recall:    0.421676

✅ Decoded DEV prediction passed the exact Phase-3 evaluator.


In [18]:
# ============================================================
# CELL 18 — Fresh Phase-1 Model4 + AdaLoRA for train+dev fractional refit
# ============================================================
gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()

BEST_EPOCH=int(best_obj["best_epoch"])
BEST_FRAC=float(best_obj.get("best_frac",BEST_EPOCH))
BEST_BATCH=int(best_obj.get("best_batch_in_epoch",0))
BEST_UPDATE=int(best_obj.get("best_global_update",0))
assert BEST_EPOCH>=1 and 0.0<BEST_FRAC<=NUM_EPOCHS

print(
    "Final refit target:",
    f"epoch_frac={BEST_FRAC:.3f}",
    f"| selected_batch={BEST_BATCH}",
    f"| selected_update={BEST_UPDATE}",
)

all_labeled_records=(
    make_labeled_records(cl_train_raw,"CL")
    +make_labeled_records(it_train_raw,"IT")
    +make_labeled_records(cl_dev_raw,"CL")
    +make_labeled_records(it_dev_raw,"IT")
)

all_counts=Counter(x["label"] for x in all_labeled_records)
N_all=len(all_labeled_records)
raw_w_all=np.array([
    math.sqrt(N_all/(NUM_LABELS*max(all_counts[lab],1)))
    for lab in LABELS
],dtype=np.float32)
raw_w_all/=raw_w_all.mean()
class_w_all=torch.tensor(
    np.clip(raw_w_all,CLASS_WEIGHT_MIN,CLASS_WEIGHT_MAX),
    dtype=torch.float32,device=DEVICE
)

def build_fresh_rr_from_phase1():
    seed_everything(SEED)
    bb=build_backbone()
    o=torch.load(PHASE1_CKPT,map_location="cpu",weights_only=False)
    st=o["model"] if isinstance(o,dict) and "model" in o else o
    bb.load_state_dict(st,strict=True)

    inject_adalora(
        bb,
        targets=ADALORA_TARGETS,
        r_init=ADALORA_R_INIT,
        alpha=ADALORA_ALPHA,
        dropout=ADALORA_DROPOUT,
    )

    m=RRClassifier(bb,NUM_LABELS,HEAD_DROPOUT)
    for n,p in m.named_parameters():
        if n.startswith(HEAD_PREFIXES):
            p.requires_grad=True

    for n,p in m.named_parameters():
        if p.requires_grad:
            assert ("adalora_" in n or n.startswith(HEAD_PREFIXES)),n
    return m.to(DEVICE)

final_model=build_fresh_rr_from_phase1()

all_ds=RRDataset(all_labeled_records,labeled=True)
all_lengths=[
    len(build_context_ids(r["domain"],r["prev"],r["target"],r["next"]))
    for r in all_labeled_records
]
all_loader=DataLoader(
    all_ds,
    batch_sampler=SortishBatchSampler(
        all_lengths,MICRO_BATCH_SIZE,seed=SEED
    ),
    collate_fn=collate_rr,
    num_workers=NUM_WORKERS,
    pin_memory=(DEVICE.type=="cuda"),
    persistent_workers=NUM_WORKERS>0,
)

final_adapter=[]
final_head=[]
for n,p in final_model.named_parameters():
    if not p.requires_grad: continue
    if "adalora_" in n: final_adapter.append(p)
    else:
        assert n.startswith(HEAD_PREFIXES),n
        final_head.append(p)

final_optimizer=make_adamw(final_adapter,final_head)

REFIT_TOTAL_BATCHES=int(
    round(BEST_FRAC*len(all_loader)/GRAD_ACCUM_STEPS)
)*GRAD_ACCUM_STEPS
REFIT_TOTAL_BATCHES=max(GRAD_ACCUM_STEPS,REFIT_TOTAL_BATCHES)

final_updates_per_epoch=math.ceil(len(all_loader)/GRAD_ACCUM_STEPS)
SCHED_EPOCHS=NUM_EPOCHS if REFIT_MATCH_ORIGINAL_SCHEDULE else max(1,math.ceil(BEST_FRAC))
final_total_updates=SCHED_EPOCHS*final_updates_per_epoch
final_warmup=max(1,int(final_total_updates*WARMUP_RATIO))

def final_lr_mult(step):
    if step<final_warmup:
        return max(1e-8,step/final_warmup)
    p=(step-final_warmup)/max(1,final_total_updates-final_warmup)
    return 0.5*(1+math.cos(math.pi*min(max(p,0.0),1.0)))

final_scheduler=torch.optim.lr_scheduler.LambdaLR(
    final_optimizer,lr_lambda=final_lr_mult
)
final_criterion=nn.CrossEntropyLoss(weight=class_w_all,label_smoothing=LABEL_SMOOTHING)
final_scaler=torch.amp.GradScaler("cuda",enabled=amp_enabled)

print(
    "All labeled records:",len(all_labeled_records),
    "| refit total micro-batches:",REFIT_TOTAL_BATCHES,
    "| loader batches/epoch:",len(all_loader),
)


Final refit target: epoch_frac=2.250 | selected_batch=526 | selected_update=2366
AdamW fused: True
All labeled records: 18987 | refit total micro-batches: 5340 | loader batches/epoch: 2374


In [19]:
# ============================================================
# CELL 19 — Final AdaLoRA refit with adaptive rank replay + ETA
# ============================================================
final_optimizer.zero_grad(set_to_none=True)
refit_start=time.time()
refit_updates=0
seen_batches=0
done=False
final_rank_history=[]

# Rank schedule is defined against the same NUM_EPOCHS horizon as heldout run.
final_schedule_total_updates=NUM_EPOCHS*final_updates_per_epoch
max_refit_epochs=int(math.ceil(REFIT_TOTAL_BATCHES/len(all_loader)))+1

for epoch in range(1,max_refit_epochs+1):
    if hasattr(all_loader.batch_sampler,"set_epoch"):
        all_loader.batch_sampler.set_epoch(epoch)

    final_model.train()
    running=torch.zeros((),device=DEVICE,dtype=torch.float32)
    n_batches=0

    for step,batch in enumerate(all_loader,1):
        if seen_batches>=REFIT_TOTAL_BATCHES:
            done=True; break

        input_ids=batch["input_ids"].to(DEVICE,non_blocking=True)
        key_valid=batch["key_valid"].to(DEVICE,non_blocking=True)
        lengths=batch["lengths"].to(DEVICE,non_blocking=True)
        pos=batch["pos"].to(DEVICE,non_blocking=True)
        labels=batch["labels"].to(DEVICE,non_blocking=True)

        with torch.amp.autocast(
            device_type="cuda",dtype=torch.float16,enabled=amp_enabled
        ):
            logits=final_model(input_ids,key_valid,lengths,pos)
            loss=final_criterion(logits,labels)
            scaled=loss/GRAD_ACCUM_STEPS

        if seen_batches%NAN_CHECK_EVERY==0:
            if not bool(torch.isfinite(loss.detach()).item()):
                raise FloatingPointError("Non-finite refit loss")

        final_scaler.scale(scaled).backward()
        running.add_(loss.detach().float())
        n_batches+=1
        seen_batches+=1

        do_update=(
            seen_batches%GRAD_ACCUM_STEPS==0
            or seen_batches>=REFIT_TOTAL_BATCHES
        )
        if do_update:
            final_scaler.unscale_(final_optimizer)
            adalora_update_importance(final_model)

            torch.nn.utils.clip_grad_norm_(
                [p for p in final_model.parameters() if p.requires_grad],
                GRAD_CLIP
            )

            old_scale=final_scaler.get_scale()
            final_scaler.step(final_optimizer)
            final_scaler.update()
            new_scale=final_scaler.get_scale()
            final_optimizer.zero_grad(set_to_none=True)

            if new_scale>=old_scale:
                final_scheduler.step()
                refit_updates+=1

                if (
                    refit_updates%ADALORA_PRUNE_INTERVAL==0
                    or refit_updates==int(
                        final_schedule_total_updates*ADALORA_TFINAL_FRAC
                    )
                ):
                    target_budget=rank_budget_for_update(
                        final_model,refit_updates,final_schedule_total_updates
                    )
                    pruned=adalora_prune_to_budget(
                        final_model,target_budget
                    )
                    _,active,maxrank,rank_stats=adalora_rank_summary(final_model)
                    final_rank_history.append({
                        "update":int(refit_updates),
                        "target_budget":int(target_budget),
                        "active":int(active),
                        "maximum":int(maxrank),
                        "rank_min":int(rank_stats["min"]),
                        "rank_mean":float(rank_stats["mean"]),
                        "rank_max":int(rank_stats["max"]),
                        "pruned_now":int(pruned),
                    })

        if seen_batches%PRINT_EVERY==0 or seen_batches>=REFIT_TOTAL_BATCHES:
            elapsed=time.time()-refit_start
            rate=elapsed/max(seen_batches,1)
            eta=rate*(REFIT_TOTAL_BATCHES-seen_batches)
            _,active,maxrank,rank_stats=adalora_rank_summary(final_model)
            print(
                f"refit batch={seen_batches}/{REFIT_TOTAL_BATCHES} "
                f"epoch≈{seen_batches/len(all_loader):.3f} "
                f"loss={float((running/max(n_batches,1)).item()):.4f} "
                f"rank={active}/{maxrank} "
                f"(min/mean/max={rank_stats['min']}/{rank_stats['mean']:.1f}/{rank_stats['max']}) "
                f"elapsed={fmt_sec(elapsed)} ETA≈{fmt_sec(eta)}",
                flush=True
            )

        if seen_batches>=REFIT_TOTAL_BATCHES:
            done=True; break

    if done: break

assert seen_batches==REFIT_TOTAL_BATCHES,(seen_batches,REFIT_TOTAL_BATCHES)

final_task_state={
    k:v.detach().cpu()
    for k,v in final_model.state_dict().items()
    if ("adalora_" in k or k.startswith(HEAD_PREFIXES))
}

rank_rows,rank_active,rank_max,rank_stats=adalora_rank_summary(final_model)
torch.save({
    "task_state":final_task_state,
    "source_best_epoch_frac":BEST_FRAC,
    "source_best_batch_in_epoch":BEST_BATCH,
    "source_best_global_update":BEST_UPDATE,
    "refit_total_batches":REFIT_TOTAL_BATCHES,
    "refit_optimizer_updates":refit_updates,
    "source_best_dev_f1":best_f1,
    "rank_history":final_rank_history,
    "rank_map":{n:int(a) for n,a,_ in rank_rows},
    "rank_stats":rank_stats,
    "labels":LABELS,
    "label_to_id":LABEL_TO_ID,
},FINAL_CKPT)

print(
    "✅ Final refit saved:",FINAL_CKPT,
    "| final rank=",f"{rank_active}/{rank_max}",
    "| elapsed:",fmt_sec(time.time()-refit_start)
)


refit batch=250/5340 epoch≈0.105 loss=2.1574 rank=1632/1632 (min/mean/max=16/16.0/16) elapsed=00:00:51 ETA≈00:17:32
refit batch=500/5340 epoch≈0.211 loss=1.9187 rank=1605/1632 (min/mean/max=10/15.7/16) elapsed=00:01:44 ETA≈00:16:54
refit batch=750/5340 epoch≈0.316 loss=1.7900 rank=1435/1632 (min/mean/max=2/14.1/16) elapsed=00:02:35 ETA≈00:15:51
refit batch=1000/5340 epoch≈0.421 loss=1.6864 rank=1337/1632 (min/mean/max=2/13.1/16) elapsed=00:03:29 ETA≈00:15:07
refit batch=1250/5340 epoch≈0.527 loss=1.6155 rank=1210/1632 (min/mean/max=2/11.9/16) elapsed=00:04:22 ETA≈00:14:18
refit batch=1500/5340 epoch≈0.632 loss=1.5611 rank=1138/1632 (min/mean/max=2/11.2/16) elapsed=00:05:15 ETA≈00:13:28
refit batch=1750/5340 epoch≈0.737 loss=1.5161 rank=1047/1632 (min/mean/max=2/10.3/16) elapsed=00:06:08 ETA≈00:12:35
refit batch=2000/5340 epoch≈0.842 loss=1.4713 rank=997/1632 (min/mean/max=2/9.8/16) elapsed=00:07:01 ETA≈00:11:44
refit batch=2250/5340 epoch≈0.948 loss=1.4313 rank=937/1632 (min/mean/max=2

In [20]:
# ============================================================
# CELL 19 — Build test sentence records
# ============================================================

def make_test_records(test_dict, domain):
    records = []
    for raw_id, sents in test_dict.items():
        did = normalize_id(raw_id)
        assert isinstance(sents, list)
        for i, sent in enumerate(sents):
            records.append({
                "doc_id": did,
                "sent_idx": i,
                "domain": domain,
                "prev": " ".join(sents[max(0, i-CONTEXT_SENTS):i]) if CONTEXT_SENTS > 0 else "",
                "target": sent,
                "next": " ".join(sents[i+1:i+1+CONTEXT_SENTS]) if CONTEXT_SENTS > 0 else "",
                "pos": position_features(i, len(sents)),
            })
    return records

test_records = (
    make_test_records(cl_test_raw, "CL")
    + make_test_records(it_test_raw, "IT")
)

test_loader = DataLoader(
    RRDataset(test_records, labeled=False),
    batch_size=EVAL_BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_rr,
    num_workers=2,
    pin_memory=True,
)

print("Total test sentence records:", len(test_records))
assert len(test_records) == (
    sum(len(v) for v in cl_test_raw.values())
    + sum(len(v) for v in it_test_raw.values())
)

Total test sentence records: 2197


In [21]:
# ============================================================
# CELL 21 — Test inference: heldout / refit / ensemble
# ============================================================
@torch.no_grad()
def predict_test_logits(m,loader):
    m.eval()
    outs,meta=[],[]
    for batch in loader:
        input_ids=batch["input_ids"].to(DEVICE,non_blocking=True)
        key_valid=batch["key_valid"].to(DEVICE,non_blocking=True)
        lengths=batch["lengths"].to(DEVICE,non_blocking=True)
        pos=batch["pos"].to(DEVICE,non_blocking=True)

        with torch.amp.autocast(
            device_type="cuda",dtype=torch.float16,enabled=amp_enabled
        ):
            lg=m(input_ids,key_valid,lengths,pos)

        outs.append(lg.float().cpu().numpy())
        meta.extend(zip(
            batch["doc_id"],batch["sent_idx"],batch["domain"]
        ))

    return np.concatenate(outs,0),meta


domain_stats_all=build_domain_stats(
    cl_train_raw+cl_dev_raw,
    it_train_raw+it_dev_raw,
)

held_logits,test_meta=predict_test_logits(model,test_loader)
refit_logits,test_meta_refit=predict_test_logits(final_model,test_loader)

assert test_meta==test_meta_refit

np.savez_compressed(
    ROOT/"test_logits.npz",
    heldout=held_logits,
    refit=refit_logits,
    doc_id=np.asarray([x[0] for x in test_meta],dtype=object),
    sent_idx=np.asarray([x[1] for x in test_meta],dtype=np.int64),
    domain=np.asarray([x[2] for x in test_meta],dtype=object),
)

def pred_from_logits(lg,name):
    ids=decode_docs(
        lg,test_meta,domain_stats_all,
        DECODE_ALPHA,DECODE_BETA
    )
    pred=records_to_prediction(test_meta,ids)
    cnt=Counter(ids.tolist())
    print(
        f"[{name}] counts:",
        {LABELS[k]:cnt[k] for k in sorted(cnt)}
    )
    return pred

predictions={
    "heldout":pred_from_logits(held_logits,"heldout"),
    "refit":pred_from_logits(refit_logits,"refit"),
}

if ENSEMBLE_HELDOUT_AND_REFIT:
    ensemble_logits=0.5*held_logits+0.5*refit_logits
    predictions["ensemble"]=pred_from_logits(
        ensemble_logits,"ensemble"
    )

prediction=predictions["refit"]

with open(OFFICIAL_PRED,"w",encoding="utf-8") as f:
    json.dump(prediction,f,ensure_ascii=False,indent=2)

print("Saved:",OFFICIAL_PRED)
print("Prediction documents:",len(prediction))


[heldout] counts: {'Fact': 415, 'Issue': 15, 'ArgumentPetitioner': 261, 'ArgumentRespondent': 43, 'Statute': 198, 'Dissent': 7, 'PrecedentReliedUpon': 369, 'PrecedentNotReliedUpon': 26, 'PrecedentOverruled': 44, 'RulingByLowerCourt': 112, 'RatioOfTheDecision': 650, 'RulingByPresentCourt': 38, 'None': 19}
[refit] counts: {'Fact': 470, 'Issue': 11, 'ArgumentPetitioner': 226, 'ArgumentRespondent': 72, 'Statute': 193, 'Dissent': 6, 'PrecedentReliedUpon': 356, 'PrecedentNotReliedUpon': 13, 'PrecedentOverruled': 32, 'RulingByLowerCourt': 161, 'RatioOfTheDecision': 602, 'RulingByPresentCourt': 41, 'None': 14}
[ensemble] counts: {'Fact': 442, 'Issue': 13, 'ArgumentPetitioner': 240, 'ArgumentRespondent': 50, 'Statute': 191, 'Dissent': 7, 'PrecedentReliedUpon': 369, 'PrecedentNotReliedUpon': 18, 'PrecedentOverruled': 24, 'RulingByLowerCourt': 125, 'RatioOfTheDecision': 664, 'RulingByPresentCourt': 38, 'None': 16}
Saved: /kaggle/working/phase3_exp5_model4_adalora/prediction.json
Prediction docume

In [22]:
# ============================================================
# CELL 22 — Strict structural checks + all submission ZIP variants
# ============================================================
def normalized_test_map(raw):
    out={}
    for raw_id,sents in raw.items():
        did=normalize_id(raw_id)
        assert did not in out,f"Duplicate normalized ID: {did}"
        out[did]=sents
    return out

cl_test=normalized_test_map(cl_test_raw)
it_test=normalized_test_map(it_test_raw)
assert not(set(cl_test)&set(it_test))

expected={}
expected.update(cl_test); expected.update(it_test)
allowed=set(LABELS)

def validate_test_prediction(pred):
    assert isinstance(pred,dict)
    assert set(pred)==set(expected),(
        f"missing={set(expected)-set(pred)}, "
        f"extra={set(pred)-set(expected)}"
    )
    for did,sents in expected.items():
        assert isinstance(pred[did],list),did
        assert len(pred[did])==len(sents),(did,len(pred[did]),len(sents))
        assert all(x in allowed for x in pred[did]),did
    return True

for name,pred in predictions.items():
    assert validate_test_prediction(pred)

    jp=ROOT/f"prediction_{name}.json"
    zp=ROOT/f"submission_{name}.zip"

    with open(jp,"w",encoding="utf-8") as f:
        json.dump(pred,f,ensure_ascii=False,indent=2)

    with zipfile.ZipFile(zp,"w",zipfile.ZIP_DEFLATED) as z:
        z.write(jp,arcname="prediction.json")

    with zipfile.ZipFile(zp,"r") as z:
        assert z.namelist()==["prediction.json"]
        chk=json.loads(z.read("prediction.json").decode("utf-8"))
    assert chk==pred
    print("✅",zp.name)

# Primary submission.zip is the refit copy; submit variants separately.
with zipfile.ZipFile(SUBMISSION_ZIP,"w",zipfile.ZIP_DEFLATED) as z:
    z.write(ROOT/"prediction_refit.json",arcname="prediction.json")
with zipfile.ZipFile(SUBMISSION_ZIP,"r") as z:
    assert z.namelist()==["prediction.json"]

assert sum(len(v) for v in prediction.values())==2197
assert len(prediction)==10

print("✅ FINAL STRUCTURAL CONTRACT PASSED")
print("Documents:",len(prediction))
print("Sentences:",sum(len(v) for v in prediction.values()))
print("submission.zip == refit")
print("Variants:",", ".join(f"submission_{k}.zip" for k in predictions))

rank_rows,rank_active,rank_max,rank_stats=adalora_rank_summary(model)
manifest={
    "experiment":"Phase3_Exp5_Model4_AdaLoRA",
    "phase1_model":"Model4_Efficient_GQA_RoPE_RMSNorm_SwiGLU",
    "technique":"AdaLoRA",
    "targets":ADALORA_TARGETS,
    "r_init":ADALORA_R_INIT,
    "r_target":ADALORA_R_TARGET,
    "alpha":ADALORA_ALPHA,
    "dropout":ADALORA_DROPOUT,
    "tinit_frac":ADALORA_TINIT_FRAC,
    "tfinal_frac":ADALORA_TFINAL_FRAC,
    "prune_interval":ADALORA_PRUNE_INTERVAL,
    "best_epoch":int(best_obj["best_epoch"]),
    "best_epoch_frac":float(best_obj.get("best_frac",best_obj["best_epoch"])),
    "best_batch_in_epoch":int(best_obj.get("best_batch_in_epoch",0)),
    "best_global_update":int(best_obj.get("best_global_update",0)),
    "best_dev_macro_f1":float(best_obj["best_macro_f1"]),
    "heldout_active_rank_components":int(rank_active),
    "heldout_max_rank_components":int(rank_max),
    "heldout_rank_map":{n:int(a) for n,a,_ in rank_rows},
    "heldout_rank_stats":rank_stats,
    "min_rank_per_module":ADALORA_MIN_RANK_PER_MODULE,
    "decode_alpha":float(DECODE_ALPHA),
    "decode_beta":float(DECODE_BETA),
    "seed":SEED,
    "tokenizer_sha256":sha256_file(TOKENIZER_FILE),
    "phase1_checkpoint":str(PHASE1_CKPT),
    "artifacts":sorted([p.name for p in ROOT.iterdir() if p.is_file()]),
}
with open(ROOT/"experiment_manifest.json","w",encoding="utf-8") as f:
    json.dump(manifest,f,indent=2)

print("✅ Saved:",ROOT/"experiment_manifest.json")


✅ submission_heldout.zip
✅ submission_refit.zip
✅ submission_ensemble.zip
✅ FINAL STRUCTURAL CONTRACT PASSED
Documents: 10
Sentences: 2197
submission.zip == refit
Variants: submission_heldout.zip, submission_refit.zip, submission_ensemble.zip
✅ Saved: /kaggle/working/phase3_exp5_model4_adalora/experiment_manifest.json


In [23]:

# ============================================================
# CELL 23 — Export teacher logits for mandatory KD experiment
# ============================================================
# We save two teacher sets:
#   1) heldout teacher logits on TRAIN only -> clean KD development run
#   2) refit teacher logits on TRAIN+DEV -> final KD refit
#
# Both are keyed by (doc_id, sent_idx, domain), so Exp6 can align them exactly.

def export_teacher_logits(m, records, filename):
    ds = RRDataset(records, labeled=True)
    loader = DataLoader(
        ds,
        batch_size=EVAL_BATCH_SIZE,
        shuffle=False,
        collate_fn=collate_rr,
        num_workers=NUM_WORKERS,
        pin_memory=(DEVICE.type=="cuda"),
    )
    logits, meta, y = predict_logits(m, loader)
    assert len(logits) == len(records) == len(meta)

    path = ROOT / filename
    np.savez_compressed(
        path,
        logits=logits.astype(np.float32),
        y=np.asarray(y, dtype=np.int64),
        doc_id=np.asarray([x[0] for x in meta], dtype=object),
        sent_idx=np.asarray([x[1] for x in meta], dtype=np.int64),
        domain=np.asarray([x[2] for x in meta], dtype=object),
    )

    acc=float((logits.argmax(-1)==np.asarray(y)).mean())
    print("✅ Saved:",path,"| rows:",len(logits),"| argmax accuracy:",round(acc,4))
    return path

# Heldout model was selected with DEV held out, so use its TRAIN logits for
# the normal KD experiment without leaking dev labels into student training.
teacher_train_path = export_teacher_logits(
    model,
    train_records,
    "teacher_logits_train_heldout.npz",
)

# Refit teacher has seen train+dev and is useful later for the student's final
# train+dev refit after hyperparameters/checkpoint position are already selected.
teacher_refit_path = export_teacher_logits(
    final_model,
    all_labeled_records,
    "teacher_logits_train_dev_refit.npz",
)

print("KD teacher artifacts ready:")
print(" ",teacher_train_path)
print(" ",teacher_refit_path)


✅ Saved: /kaggle/working/phase3_exp5_model4_adalora/teacher_logits_train_heldout.npz | rows: 16845 | argmax accuracy: 0.9485
✅ Saved: /kaggle/working/phase3_exp5_model4_adalora/teacher_logits_train_dev_refit.npz | rows: 18987 | argmax accuracy: 0.9428
KD teacher artifacts ready:
  /kaggle/working/phase3_exp5_model4_adalora/teacher_logits_train_heldout.npz
  /kaggle/working/phase3_exp5_model4_adalora/teacher_logits_train_dev_refit.npz


## What to record for Experiment 5

Keep:
- `best_adalora_rr.pt`
- `final_refit_adalora_rr.pt`
- `submission_heldout.zip`
- `submission_refit.zip`
- `submission_ensemble.zip`
- `submission.zip`
- `history.json`
- `rank_history.json`
- `experiment_manifest.json`
- `dev_logits.npz`
- `test_logits.npz`
- completed notebook

For the report record:
- Phase-1 backbone: **Model 4 efficient decoder**
- technique: **AdaLoRA**
- targets: Q, V, O + SwiGLU gate/up/down
- initial rank = 16
- target average rank budget = 8
- adaptive global pruning by EMA of AdaLoRA-style sensitivity `|e·∇e| + mean|A·∇A| + mean|B·∇B|` (no orthogonality regulariser)
- all pretrained Phase-1 weights frozen
- best combined dev Macro-F1 and exact epoch fraction
- active rank distribution at the best checkpoint
- minimum active rank per adapted projection = 2 (safety floor)
- Codabench heldout / refit / ensemble scores

### Run order
1. Attach `eff_full_best*.pt` + exact `tokenizer.json`; T4 GPU.
2. Run top-to-bottom.
3. Confirm `strict=True` Phase-1 checkpoint load succeeds.
4. Confirm **102 AdaLoRA modules** are injected (`17 × 6`).
5. Confirm initial active-rank components = `102 × 16 = 1632`.
6. Watch the printed rank budget shrink during training.
7. Exact Phase-3 evaluator must pass before final refit.
8. Continue through dev decoding → fractional refit → test inference → structural checks.
9. If submission budget allows, submit heldout, refit, and ensemble separately and retain the best Codabench score as Exp5.

- `teacher_logits_train_heldout.npz` — use for Exp6 train-only KD development.
- `teacher_logits_train_dev_refit.npz` — use only for Exp6 final train+dev refit after model selection.
